## 🚀 Bootstrap: Khởi tạo môi trường nhanh

Chạy cell này để tự động thiết lập cấu trúc thư mục trên Google Drive và tải mã nguồn dự án.

**Lưu ý:** Bạn cần cấp quyền truy cập Google Drive khi được yêu cầu.

In [ ]:
import os
from google.colab import drive
from pathlib import Path

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Cấu hình Link (Nếu có)
# Vì bạn gửi Data ZIP qua Gmail, cộng tác viên sẽ tự giải nén vào thư mục clinical_asr_vimedcss.
MODELS_ZIP_URL = 'LINK_MODELS_ZIP_CỦA_BẠN'

REPO_URL = "https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant"
PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/clinical_asr_vimedcss")
DRIVE_MODELS_ROOT = Path("/content/drive/MyDrive/clinical_asr_models")

# 3. Khởi tạo cấu trúc thư mục
print("[INFO] Đang kiểm tra cấu trúc thư mục trên Drive...")
for path in [DRIVE_DATA_ROOT, DRIVE_MODELS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def setup_models(zip_url, target_path):
    if not zip_url or zip_url == 'LINK_MODELS_ZIP_CỦA_BẠN':
        print("[SKIP] Không có link tải Models Checkpoint.")
        return

    print(f"[INFO] Đang tải Models Checkpoint...")
    zip_tmp = "/content/models_checkpoint.zip"
    !gdown --fuzzy {zip_url} -O {zip_tmp}

    if os.path.exists(zip_tmp):
        print(f"[INFO] Đang giải nén vào {target_path}...")
        !unzip -o -q {zip_tmp} -d /content/tmp_extract_models
        !cp -r /content/tmp_extract_models/* {target_path.parent}/
        !rm -rf /content/tmp_extract_models {zip_tmp}
        print("[OK] Hoàn tất thiết lập Models.")

# Tải models nếu cần
setup_models(MODELS_ZIP_URL, DRIVE_MODELS_ROOT)

# 4. Clone/Update Repo mã nguồn
if not PROJECT_ROOT.exists():
    print("[INFO] Đang clone repository...")
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print("[INFO] Repository đã tồn tại, đang cập nhật mã nguồn mới nhất...")
    !git -C {PROJECT_ROOT} pull

# 5. Cài đặt các thư viện cần thiết
print("[INFO] Cài đặt dependencies...")
!pip install -q datasets huggingface_hub soundfile transformers

print("\n[DONE] Hệ thống đã sẵn sàng!")
print(f"-> Lưu ý: Hãy giải nén file data bạn nhận qua Gmail vào thư mục: {DRIVE_DATA_ROOT}")

## 0. Runtime setup


Kết nối với Google Drive để lưu trữ và truy xuất dữ liệu.

In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


Thiết lập các đường dẫn thư mục chính cho dự án và tạo các thư mục cần thiết.

In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
DRIVE_ROOT = Path("/content/drive/MyDrive/clinical_asr_vimedcss")
WORK_ROOT = Path("/content/vimedcss_work")
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"
WEEK5_CKPT = Path("/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint")

for p in [DRIVE_ROOT, WORK_ROOT, AUDIO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VIMEDCSS_ROOT:", VIMEDCSS_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("WEEK5_CKPT:", WEEK5_CKPT)


PROJECT_ROOT: /content/Clinical-Ambient-Documentation-Assistant
VIMEDCSS_ROOT: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss
DRIVE_ROOT: /content/drive/MyDrive/clinical_asr_vimedcss
WORK_ROOT: /content/vimedcss_work
AUDIO_DIR: /content/Clinical-Ambient-Documentation-Assistant/data/vimedcss_audio
WEEK5_CKPT: /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint


Cấu hình mã thông báo Hugging Face để truy cập các tập dữ liệu.

In [3]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get("HF_TOKEN")
except Exception:
    token_from_secret = None

HF_TOKEN = token_from_secret or os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("[OK] HF_TOKEN is available in environment.")
else:
    print("[WARN] HF_TOKEN is not set. Public dataset access may still work, but private/rate-limited operations may fail.")


[OK] HF_TOKEN is available in environment.


Kiểm tra trạng thái hệ thống bao gồm GPU, dung lượng đĩa và phiên bản Python.

In [4]:
%%bash
set -euo pipefail

echo "=== GPU status ==="
nvidia-smi || true

echo
echo "=== Disk status ==="
df -h

echo
echo "=== Python ==="
python --version


=== GPU status ===
Tue Jun 16 12:13:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

Cập nhật mã nguồn từ GitHub và cài đặt các thư viện phụ thuộc.

In [6]:
%%bash
set -euo pipefail

REPO_URL="https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant"
PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"

log() {
  echo
  echo "============================================================"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1"
  echo "============================================================"
}

cd /content

log "STEP 1/6 — Checking repository folder"

if [ -d "$PROJECT_ROOT/.git" ]; then
  echo "[INFO] Git repo already exists at:"
  echo "$PROJECT_ROOT"

  echo
  echo "[INFO] Current git status:"
  git -C "$PROJECT_ROOT" status --short || true

  echo
  echo "[INFO] Pulling latest changes with --ff-only if possible..."
  if git -C "$PROJECT_ROOT" pull --ff-only; then
    echo "[OK] Pull completed."
  else
    echo "[WARN] Pull failed or not fast-forward. Continuing with existing repo."
  fi

elif [ -d "$PROJECT_ROOT" ]; then
  echo "[WARN] Folder exists but is not a git repo:"
  echo "$PROJECT_ROOT"

  echo
  echo "[INFO] Folder preview:"
  ls -lah "$PROJECT_ROOT" | head -30 || true

  BACKUP_DIR="${PROJECT_ROOT}_backup_$(date +%Y%m%d_%H%M%S)"

  echo
  echo "[WARN] Renaming old folder to:"
  echo "$BACKUP_DIR"

  mv "$PROJECT_ROOT" "$BACKUP_DIR"

  echo
  echo "[INFO] Cloning fresh repo with progress..."
  git clone --progress "$REPO_URL" "$PROJECT_ROOT"

else
  echo "[INFO] Repo folder does not exist."
  echo "[INFO] Cloning repo with progress..."
  git clone --progress "$REPO_URL" "$PROJECT_ROOT"
fi

log "STEP 2/6 — Entering project root"

cd "$PROJECT_ROOT"

echo "[INFO] Current directory:"
pwd

echo
echo "[INFO] Git branch and latest commit:"
git branch --show-current || true
git log -1 --oneline || true

log "STEP 3/6 — Checking Python and pip"

python --version
python -m pip --version

log "STEP 4/6 — Installing requirements.txt"

if [ -f "requirements.txt" ]; then
  echo "[INFO] Found requirements.txt"
  echo "[INFO] Installing dependencies. This may take several minutes..."

  if python -m pip install -r requirements.txt --progress-bar on; then
    echo "[OK] requirements.txt installed."
  else
    echo "[WARN] Some packages from requirements.txt failed to install."
    echo "[WARN] Continuing because Phase 0/1/2 may not need every package."
  fi
else
  echo "[WARN] requirements.txt not found. Skipping."
fi

log "STEP 5/6 — Installing Phase 0/1/2 utilities"

python -m pip install -U \
  datasets \
  huggingface_hub \
  soundfile \
  pandas \
  tqdm \
  --progress-bar on

echo "[OK] Phase 0/1/2 utilities installed."

log "STEP 6/6 — Creating required directories"

mkdir -pv \
  experiments/asr/vimedcss/manifests/raw \
  experiments/asr/vimedcss/data_audit \
  experiments/asr/vimedcss/preprocessing/subset_archives \
  data/vimedcss_audio \
  /content/vimedcss_work/hf_cache

echo
echo "[INFO] Directory check:"
ls -ld \
  experiments/asr/vimedcss/manifests/raw \
  experiments/asr/vimedcss/data_audit \
  experiments/asr/vimedcss/preprocessing/subset_archives \
  data/vimedcss_audio \
  /content/vimedcss_work/hf_cache

echo
echo "[INFO] Disk usage:"
df -h /content /content/drive 2>/dev/null || df -h /content

echo
echo "[DONE] Repo setup completed successfully."
echo "PROJECT_ROOT=$PROJECT_ROOT"


[2026-06-16 12:13:27] STEP 1/6 — Checking repository folder
[INFO] Repo folder does not exist.
[INFO] Cloning repo with progress...

[2026-06-16 12:13:29] STEP 2/6 — Entering project root
[INFO] Current directory:
/content/Clinical-Ambient-Documentation-Assistant

[INFO] Git branch and latest commit:
main
db88dbd Rename ViMedCSS preprocessing notebook to ViMedCSS_Handling.ipynb

[2026-06-16 12:13:29] STEP 3/6 — Checking Python and pip
Python 3.12.13
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)

[2026-06-16 12:13:29] STEP 4/6 — Installing requirements.txt
[INFO] Found requirements.txt
[INFO] Installing dependencies. This may take several minutes...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished wit

Cloning into '/content/Clinical-Ambient-Documentation-Assistant'...
remote: Enumerating objects: 1321, done.        
remote: Counting objects: 100% (234/234), done.        
remote: Compressing objects: 100% (142/142), done.        
remote: Total 1321 (delta 113), reused 192 (delta 78), pack-reused 1087 (from 1)        
Receiving objects: 100% (1321/1321), 10.03 MiB | 14.08 MiB/s, done.
Resolving deltas: 100% (662/662), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.

## Phase 0 — Readiness, raw manifests, schema check


Khởi tạo cấu trúc thư mục cho giai đoạn mới và kiểm tra tính sẵn sàng của môi trường.

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WEEK5_CKPT="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"

cd "$PROJECT_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

echo "[INFO] Using script dir: $SCRIPT_DIR"

bash "$SCRIPT_DIR/00_prepare_vimedcss_day1.sh" \
  "$PROJECT_ROOT" \
  "$DRIVE_ROOT" \
  "$WEEK5_CKPT"

python "$SCRIPT_DIR/00_validate_vimedcss_day1_readiness.py" \
  --project_root "$PROJECT_ROOT" \
  --drive_root "$DRIVE_ROOT" \
  --week5_checkpoint "$WEEK5_CKPT"


[INFO] Using script dir: /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss
[DONE] Week 6 Day 1 ViMedCSS skeleton prepared.
[INFO] Project root:    /content/Clinical-Ambient-Documentation-Assistant
[INFO] ViMedCSS root:   /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss
[INFO] Drive root:      /content/drive/MyDrive/clinical_asr_vimedcss
[INFO] Week5 ckpt:      /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
[NEXT] Run: python scripts/vimedcss/00_validate_vimedcss_day1_readiness.py --project_root "/content/Clinical-Ambient-Documentation-Assistant" --drive_root "/content/drive/MyDrive/clinical_asr_vimedcss" --week5_checkpoint "/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"
{
  "decision": "PASS_WITH_WARNINGS",
  "hard_failures": [],
  "warnings": [
    "manifest_missing_audio_column:train",
    "manifest_missing_segment_text:train",
    "manifest_missing_audio_column:validation"

### Phase 0A — Rebuild raw manifests from Hugging Face

Chạy cell debug 5 dòng trước. Nếu output hợp lý thì chạy cell full ở ngay sau.

Cell này giữ nguyên logic rebuild qua script:

```text
00_rebuild_vimedcss_raw_manifests_from_hf.py
```

Không download toàn bộ audio. Nó tạo raw manifests cho các split official.


Thiết lập biến môi trường cho mã thông báo Hugging Face trong Python.

In [ ]:
import os
from google.colab import userdata

# Lấy token từ phần Secrets (nhớ bật toggle cấp quyền truy cập cho notebook)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Chạy thử nghiệm việc tạo tệp manifest với số lượng dòng giới hạn để kiểm tra lỗi.

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"
CACHE_DIR="/content/vimedcss_work/hf_cache"

cd "$PROJECT_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

mkdir -p "$CACHE_DIR"

echo "[INFO] Debug rebuild with max_rows=5"
python "$SCRIPT_DIR/00_rebuild_vimedcss_raw_manifests_from_hf.py" \
  --dataset_id tensorxt/ViMedCSS \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --cache_dir "$CACHE_DIR" \
  --splits train validation test hard \
  --max_rows 5 \
  --overwrite

echo
echo "[INFO] Debug raw manifest line counts"
wc -l "$VIMEDCSS_ROOT"/manifests/raw/vimedcss_*_raw.jsonl


[INFO] Debug rebuild with max_rows=5
[INFO] Loading dataset split=train, streaming=True
[INFO] split=train columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs_terms_list', 'cs_terms_count', 'topic', 'original_video_link', 'original_video_title', 'start_time', 'end_time']
[INFO] split=train inferred={'audio_col': 'audio', 'text_col': 'segment_text', 'duration_col': 'duration_seconds', 'id_col': 'segment_id'}
[INFO] Loading dataset split=validation, streaming=True
[INFO] split=validation columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs_terms_list', 'cs_terms_count', 'topic', 'original_video_link', 'original_video_title', 'start_time', 'end_time']
[INFO] split=validation inferred={'audio_col': 'audio', 'text_col': 'segment_text', 'duration_col': 'duration_seconds', 'id_col': 'segment_id'}
[INFO] Loading dataset split=test, streaming=True
[INFO] split=test columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs_terms_list', 'cs_ter

Fatal Python error: PyGILState_Release: thread state 0x785f92092a20 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007860a98bc000 (most recent call first):
  <no Python frame>

Extension modules: zstandard.backend_c, numpy._core._multiarray_umath, numpy._core._multiarray_tests, numpy.linalg._umath_linalg, pyarrow.lib, numpy.random._common, numpy.random.bit_generator, numpy.random._bounded_integers, numpy.random._mt19937, numpy.random.mtrand, numpy.random._philox, numpy.random._pcg64, numpy.random._sfc64, numpy.random._generator, _cyutility, pandas._libs._cyutility, pandas._libs.tslibs.ccalendar, pandas._libs.tslibs.np_datetime, pandas._libs.tslibs.dtypes, pandas._libs.tslibs.base, pandas._libs.tslibs.nattype, pandas._libs.tslibs.timezones, pandas._libs.properties, pandas._libs.tslibs.fields, pandas._libs.tslibs.timedeltas, pandas._libs.tslibs.tzconversion, pandas._libs.tslibs.timestamps, pandas._libs.tslibs.offsets, pandas._libs.

CalledProcessError: Command 'b'set -euo pipefail\n\nPROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"\nVIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"\nCACHE_DIR="/content/vimedcss_work/hf_cache"\n\ncd "$PROJECT_ROOT"\n\nSCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"\nif [ ! -d "$SCRIPT_DIR" ]; then\n  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"\nfi\n\nmkdir -p "$CACHE_DIR"\n\necho "[INFO] Debug rebuild with max_rows=5"\npython "$SCRIPT_DIR/00_rebuild_vimedcss_raw_manifests_from_hf.py" \\\n  --dataset_id tensorxt/ViMedCSS \\\n  --vimedcss_root "$VIMEDCSS_ROOT" \\\n  --cache_dir "$CACHE_DIR" \\\n  --splits train validation test hard \\\n  --max_rows 5 \\\n  --overwrite\n\necho\necho "[INFO] Debug raw manifest line counts"\nwc -l "$VIMEDCSS_ROOT"/manifests/raw/vimedcss_*_raw.jsonl\n'' returned non-zero exit status 134.

Liệt kê các tệp manifest thô vừa được tạo trong thư mục tương ứng.

In [ ]:
%%bash
ls -l /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw

total 32
-rw-r--r-- 1 root root 4896 Jun 15 18:06 vimedcss_hard_raw.jsonl
-rw-r--r-- 1 root root 4763 Jun 15 18:06 vimedcss_test_raw.jsonl
-rw-r--r-- 1 root root 4841 Jun 15 18:06 vimedcss_train_raw.jsonl
-rw-r--r-- 1 root root 4941 Jun 15 18:06 vimedcss_validation_raw.jsonl


Kiểm tra nội dung chi tiết của một dòng dữ liệu trong các tệp manifest.

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"

for split in ["train", "validation", "test", "hard"]:
    p = VIMEDCSS_ROOT / "manifests/raw" / f"vimedcss_{split}_raw.jsonl"
    print(f"\n========== {split} ==========")
    print("path:", p)
    if not p.exists():
        print("MISSING")
        continue
    with p.open("r", encoding="utf-8") as f:
        line = next((x for x in f if x.strip()), None)
    if line is None:
        print("EMPTY")
        continue
    row = json.loads(line)
    print("columns:", sorted(row.keys()))
    for k, v in row.items():
        print(f"{k}: {str(v)[:300]}")



========== train ==========
path: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_train_raw.jsonl
columns: ['_audio_bytes_present', '_audio_type', '_created_at', '_source_audio_col', '_source_duration_col', '_source_id_col', '_source_text_col', 'audio', 'audio_path', 'cs_terms_count', 'cs_terms_list', 'dataset_id', 'duration_seconds', 'end_time', 'original_video_link', 'original_video_title', 'row_index', 'sample_id', 'sampling_rate', 'segment_id', 'segment_text', 'split', 'start_time', 'topic']
sample_id: vimedcss_train_000000
segment_id: Med_CS-0-1
dataset_id: tensorxt/ViMedCSS
split: train
row_index: 0
audio: Med_CS-0-1.wav
audio_path: Med_CS-0-1.wav
segment_text: 5 alpha reductase là một enzyme chuyển đổi hóc môn nam testosterone thành dạng có hiệu lực hơn được gọi là dihydrotestosterone.
duration_seconds: 10
sampling_rate: None
cs_terms_list: ['reductase', 'testosteron']
cs_terms_count: 2
topic: Medical Sciences
original_video_lin

Thực hiện tạo toàn bộ tệp manifest cho tất cả các phần dữ liệu (train, validation, test, hard).

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"
CACHE_DIR="/content/vimedcss_work/hf_cache"

cd "$PROJECT_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

mkdir -p "$CACHE_DIR"

echo "[INFO] Rebuilding full raw manifests for official splits."
python "$SCRIPT_DIR/00_rebuild_vimedcss_raw_manifests_from_hf.py" \
  --dataset_id tensorxt/ViMedCSS \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --cache_dir "$CACHE_DIR" \
  --splits train validation test hard \
  --overwrite

echo
echo "[INFO] Full raw manifest line counts"
wc -l "$VIMEDCSS_ROOT"/manifests/raw/vimedcss_*_raw.jsonl


[INFO] Rebuilding full raw manifests for official splits.
[INFO] Loading dataset split=train, streaming=True
[INFO] split=train columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs_terms_list', 'cs_terms_count', 'topic', 'original_video_link', 'original_video_title', 'start_time', 'end_time']
[INFO] split=train inferred={'audio_col': 'audio', 'text_col': 'segment_text', 'duration_col': 'duration_seconds', 'id_col': 'segment_id'}
[INFO] Loading dataset split=validation, streaming=True
[INFO] split=validation columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs_terms_list', 'cs_terms_count', 'topic', 'original_video_link', 'original_video_title', 'start_time', 'end_time']
[INFO] split=validation inferred={'audio_col': 'audio', 'text_col': 'segment_text', 'duration_col': 'duration_seconds', 'id_col': 'segment_id'}
[INFO] Loading dataset split=test, streaming=True
[INFO] split=test columns=['segment_id', 'audio', 'duration_seconds', 'segment_text', 'cs

Chuẩn hóa định dạng các cột trong tệp manifest để đảm bảo tính nhất quán.

In [ ]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
MANIFEST_DIR = VIMEDCSS_ROOT / "manifests/raw"
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"

for split in ["train", "validation", "test", "hard"]:
    p = MANIFEST_DIR / f"vimedcss_{split}_raw.jsonl"
    if not p.exists():
        print(f"[WARN] Missing: {p}")
        continue

    df = pd.read_json(p, lines=True)

    if "audio" not in df.columns and "audio_path" in df.columns:
        df["audio"] = df["audio_path"]

    if "segment_text" not in df.columns:
        for cand in ["text", "sentence", "transcript_text", "reference_text"]:
            if cand in df.columns:
                df["segment_text"] = df[cand]
                break

    if "audio" in df.columns:
        if "audio_original" not in df.columns:
            df["audio_original"] = df["audio"]
        df["audio_basename"] = df["audio"].apply(lambda x: os.path.basename(str(x)))
        df["clean_audio_path"] = df["audio_basename"].apply(lambda x: str(AUDIO_DIR / x))

    df.to_json(p, orient="records", lines=True, force_ascii=False)
    print(f"[OK] Standardized {split}: {len(df)} rows -> {p}")


/tmp/ipykernel_2501/333633839.py:33: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


[OK] Standardized train: 11832 rows -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_train_raw.jsonl


/tmp/ipykernel_2501/333633839.py:33: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


[OK] Standardized validation: 1714 rows -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_validation_raw.jsonl
[OK] Standardized test: 1614 rows -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_test_raw.jsonl
[OK] Standardized hard: 658 rows -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_hard_raw.jsonl


/tmp/ipykernel_2501/333633839.py:33: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)
/tmp/ipykernel_2501/333633839.py:33: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


Kiểm tra lại lần cuối tính sẵn sàng của dữ liệu sau khi đã chuẩn hóa.

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WEEK5_CKPT="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"

cd "$PROJECT_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

python -u "$SCRIPT_DIR/00_validate_vimedcss_day1_readiness.py" \
  --project_root "$PROJECT_ROOT" \
  --drive_root "$DRIVE_ROOT" \
  --week5_checkpoint "$WEEK5_CKPT"


{
  "decision": "PASS_WITH_WARNINGS",
  "hard_failures": [],
  "warnings": [
    "week5_checkpoint_missing_or_incomplete"
  ],
  "markdown": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.md",
  "json": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.json"
}


Nén các tệp script và báo cáo quan trọng thành định dạng tar.gz.

In [ ]:
import tarfile
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
OUT = Path("/content/week6_phase0_light_artifacts.tar.gz")

candidates = [
    PROJECT_ROOT / "scripts/asr_vimedcss/00_prepare_vimedcss_day1.sh",
    PROJECT_ROOT / "scripts/asr_vimedcss/00_validate_vimedcss_day1_readiness.py",
    PROJECT_ROOT / "scripts/asr_vimedcss/00_rebuild_vimedcss_raw_manifests_from_hf.py",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/VIMEDCSS_TASK_BRIEF.md",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/COLAB_CPU_GPU_PROTOCOL.md",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/DRIVE_STORAGE_POLICY.md",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/TTS_DATA_REQUEST_MESSAGE.md",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/WEEK6_ENVIRONMENT_CONFIG.json",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.md",
    PROJECT_ROOT / "experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.json",
    PROJECT_ROOT / "experiments/asr/vimedcss/preprocessing/PREPROCESSING_POLICY_VIMEDCSS.md",
]

with tarfile.open(OUT, "w:gz") as tar:
    for p in candidates:
        if p.exists():
            tar.add(p, arcname=str(p.relative_to(PROJECT_ROOT)))
            print("[ADD]", p.relative_to(PROJECT_ROOT))
        else:
            print("[SKIP missing]", p)

print("Created:", OUT)


[ADD] scripts/asr_vimedcss/00_prepare_vimedcss_day1.sh
[ADD] scripts/asr_vimedcss/00_validate_vimedcss_day1_readiness.py
[ADD] scripts/asr_vimedcss/00_rebuild_vimedcss_raw_manifests_from_hf.py
[ADD] experiments/asr/vimedcss/reports/VIMEDCSS_TASK_BRIEF.md
[ADD] experiments/asr/vimedcss/reports/COLAB_CPU_GPU_PROTOCOL.md
[ADD] experiments/asr/vimedcss/reports/DRIVE_STORAGE_POLICY.md
[ADD] experiments/asr/vimedcss/reports/TTS_DATA_REQUEST_MESSAGE.md
[ADD] experiments/asr/vimedcss/reports/WEEK6_ENVIRONMENT_CONFIG.json
[ADD] experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.md
[ADD] experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.json
[ADD] experiments/asr/vimedcss/preprocessing/PREPROCESSING_POLICY_VIMEDCSS.md
Created: /content/week6_phase0_light_artifacts.tar.gz


### Download Phase 0 Artifacts

Nén toàn bộ các tệp quan trọng bao gồm Raw Manifests, Reports và Scripts vào một file zip duy nhất.

Gom tất cả kết quả của Giai đoạn 0 vào một tệp ZIP duy nhất để tải về.

In [ ]:
import os
from google.colab import files

ZIP_NAME = "ViMedCSS_Phase0_Artifacts.zip"

# Gom các đường dẫn quan trọng
PROJECT_ROOT = "/content/Clinical-Ambient-Documentation-Assistant"
PATHS_TO_ZIP = [
    f"{PROJECT_ROOT}/experiments/asr/vimedcss/manifests/raw",
    f"{PROJECT_ROOT}/experiments/asr/vimedcss/reports",
    f"{PROJECT_ROOT}/scripts/asr_vimedcss",
    "/content/week6_phase0_light_artifacts.tar.gz"
]

# Thực hiện nén bằng lệnh shell
paths_str = " ".join(PATHS_TO_ZIP)
!zip -r {ZIP_NAME} {paths_str}

print(f"\n[DONE] Đã tạo file: {ZIP_NAME}")
# files.download(ZIP_NAME) # Bỏ comment dòng này nếu muốn tự động tải về ngay

  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/ (stored 0%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_hard_raw.jsonl (deflated 86%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_validation_raw.jsonl (deflated 87%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_train_raw.jsonl (deflated 89%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_test_raw.jsonl (deflated 86%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/reports/ (stored 0%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/reports/WEEK6_EXECUTION_READINESS_CHECK.md (deflated 67%)
  adding: content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/rep

Kích hoạt trình duyệt để tải tệp nén kết quả về máy tính cá nhân.

In [ ]:
from google.colab import files

try:
    files.download('ViMedCSS_Phase0_Artifacts.zip')
    print("Đang khởi tạo quá trình tải về...")
except Exception as e:
    print(f"Lỗi khi tải file: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang khởi tạo quá trình tải về...


## Phase 1 — Sample-based data audit


Thực hiện quy trình kiểm tra dữ liệu dựa trên mẫu (Sample-based audit) để phát hiện các lỗi tiềm ẩn trong tệp manifest.

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"

cd "$PROJECT_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

echo "[INFO] Running Phase 1 audit."
python "$SCRIPT_DIR/01_audit_vimedcss_manifests.py" \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --output_dir "$VIMEDCSS_ROOT/data_audit"


[INFO] Running Phase 1 audit.
{
  "status": "DONE",
  "output_dir": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit",
  "report": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/AUDIT_VIMEDCSS_SAMPLE_BASED_REPORT.md",
  "summary_csv": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vimedcss_audit_summary.csv",
  "samples_csv": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vimedcss_audit_samples.csv",
  "suspect_csv": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vimedcss_suspect_samples.csv",
  "summary_json": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vimedcss_audit_summary.json"
}


Hiển thị danh sách các tệp tin kết quả, thống kê số dòng và xem trước nội dung báo cáo kiểm tra dữ liệu.

In [ ]:
%%bash
# Không sử dụng pipefail cho các lệnh có dùng 'head' để tránh lỗi Broken Pipe (status 32)
set -eu

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"

echo "=== Phase 1 outputs ==="
ls -lh "$VIMEDCSS_ROOT/data_audit"

echo
echo "=== CSV line counts ==="
wc -l "$VIMEDCSS_ROOT/data_audit/vimedcss_audit_summary.csv"
wc -l "$VIMEDCSS_ROOT/data_audit/vimedcss_audit_samples.csv"
wc -l "$VIMEDCSS_ROOT/data_audit/vimedcss_suspect_samples.csv"

echo
echo "=== Summary JSON preview ==="
# Tạm thời tắt errexit hoặc tránh pipefail để lệnh head không làm treo cell
python -m json.tool "$VIMEDCSS_ROOT/data_audit/vimedcss_audit_summary.json" 2>/dev/null | head -n 120 || true

echo
echo "=== Report preview ==="
sed -n '1,220p' "$VIMEDCSS_ROOT/data_audit/AUDIT_VIMEDCSS_SAMPLE_BASED_REPORT.md"

=== Phase 1 outputs ===
total 4.8M
-rw-r--r-- 1 root root 4.0K Jun 15 18:30 AUDIT_VIMEDCSS_SAMPLE_BASED_REPORT.md
-rw-r--r-- 1 root root 1.1K Jun 15 18:14 REBUILD_VIMEDCSS_RAW_MANIFESTS_REPORT.md
-rw-r--r-- 1 root root 3.3K Jun 15 18:14 REBUILD_VIMEDCSS_RAW_MANIFESTS_SUMMARY.json
-rw-r--r-- 1 root root 4.7M Jun 15 18:30 vimedcss_audit_samples.csv
-rw-r--r-- 1 root root 3.8K Jun 15 18:30 vimedcss_audit_summary.csv
-rw-r--r-- 1 root root 7.3K Jun 15 18:30 vimedcss_audit_summary.json
-rw-r--r-- 1 root root  37K Jun 15 18:30 vimedcss_cs_term_summary.csv
-rw-r--r-- 1 root root  14K Jun 15 18:30 vimedcss_schema_examples.json
-rw-r--r-- 1 root root 2.1K Jun 15 18:30 vimedcss_suspect_samples.csv
-rw-r--r-- 1 root root  541 Jun 15 18:30 vimedcss_topic_summary.csv

=== CSV line counts ===
5 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vimedcss_audit_summary.csv
15819 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/data_audit/vim

Phân tích và liệt kê các lý do hàng đầu khiến mẫu dữ liệu bị đánh dấu là nghi ngờ (suspect).

In [ ]:
# Kiểm tra top suspect reasons.
# Nếu suspect quá cao, dừng để xem lại raw manifests/schema trước khi sang Phase 2.

import csv
from collections import Counter
from pathlib import Path

VIMEDCSS_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss")
suspect_csv = VIMEDCSS_ROOT / "data_audit/vimedcss_suspect_samples.csv"

counter = Counter()

if suspect_csv.exists():
    with suspect_csv.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            for reason in (row.get("suspect_reasons") or "").split(";"):
                reason = reason.strip()
                if reason:
                    counter[reason] += 1

print("Top suspect reasons:")
if counter:
    for reason, count in counter.most_common(30):
        print(f"{count:8d} | {reason}")
else:
    print("No suspect reasons found.")


Top suspect reasons:
       3 | too_few_words_le_1
       3 | duration_too_long_gt_25s


Sao lưu toàn bộ kết quả kiểm tra dữ liệu từ thư mục làm việc sang Google Drive để lưu trữ lâu dài.

In [ ]:
%%bash
set -euo pipefail

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"

mkdir -p "$DRIVE_ROOT/data_audit"

echo "[INFO] Copying Phase 1 audit outputs to Drive."
cp -r "$VIMEDCSS_ROOT/data_audit/"* "$DRIVE_ROOT/data_audit/"

echo
echo "=== Drive data_audit ==="
ls -lh "$DRIVE_ROOT/data_audit"


[INFO] Copying Phase 1 audit outputs to Drive.

=== Drive data_audit ===
total 4.8M
-rw------- 1 root root 4.0K Jun 15 18:31 AUDIT_VIMEDCSS_SAMPLE_BASED_REPORT.md
-rw------- 1 root root 1.1K Jun 15 18:31 REBUILD_VIMEDCSS_RAW_MANIFESTS_REPORT.md
-rw------- 1 root root 3.3K Jun 15 18:31 REBUILD_VIMEDCSS_RAW_MANIFESTS_SUMMARY.json
-rw------- 1 root root 4.7M Jun 15 18:31 vimedcss_audit_samples.csv
-rw------- 1 root root 3.8K Jun 15 18:31 vimedcss_audit_summary.csv
-rw------- 1 root root 7.3K Jun 15 18:31 vimedcss_audit_summary.json
-rw------- 1 root root  37K Jun 15 18:31 vimedcss_cs_term_summary.csv
-rw------- 1 root root  14K Jun 15 18:31 vimedcss_schema_examples.json
-rw------- 1 root root 2.1K Jun 15 18:31 vimedcss_suspect_samples.csv
-rw------- 1 root root  541 Jun 15 18:31 vimedcss_topic_summary.csv


Đóng gói các kết quả kiểm tra Phase 1 thành tệp nén để chuẩn bị cho việc tải về máy cá nhân.

In [ ]:
# Bundle Phase 1 audit artifacts for download to local machine.

import tarfile
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
OUT = Path("/content/phase1_vimedcss_data_audit_artifacts.tar.gz")
DATA_AUDIT = PROJECT_ROOT / "experiments/asr/vimedcss/data_audit"

with tarfile.open(OUT, "w:gz") as tar:
    if DATA_AUDIT.exists():
        tar.add(DATA_AUDIT, arcname="experiments/asr/vimedcss/data_audit")
    else:
        raise FileNotFoundError(DATA_AUDIT)

print("Created:", OUT)


Created: /content/phase1_vimedcss_data_audit_artifacts.tar.gz


In [ ]:
from google.colab import files

try:
    files.download('/content/phase1_vimedcss_data_audit_artifacts.tar.gz')
    print("Đang tải xuống kết quả Phase 1...")
except Exception as e:
    print(f"Lỗi: {e}")

## Phase 2 — Manifest engineering and subset materialization


### Phase 2A — Create run manifests first

Cell này chạy `--manifests_only` để kiểm tra subset selection trước. Nó không tạo archive audio thật.

Kỳ vọng chính:

```text
run0_eval_val_200.jsonl       200
run0_eval_hard_200.jsonl      200
run0_eval_test_200.jsonl      200
runA_train_200.jsonl          200
runA_eval_val_100.jsonl       100
runB_train_1000.jsonl         1000
runB_eval_val_200.jsonl       200
runB_eval_hard_200.jsonl      200
runC_train_4h.jsonl           khoảng 3–5 giờ audio
runC_eval_val_200.jsonl       200
runC_eval_hard_200.jsonl      200
```


In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WORK_ROOT="/content/vimedcss_work"

cd "$PROJECT_ROOT"
mkdir -p "$WORK_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

echo "[INFO] Cleaning previous run manifests, keeping raw manifests."
rm -rf "$VIMEDCSS_ROOT/manifests/run0" \
       "$VIMEDCSS_ROOT/manifests/runA" \
       "$VIMEDCSS_ROOT/manifests/runB" \
       "$VIMEDCSS_ROOT/manifests/runC"

echo "[INFO] Running Phase 2 manifests_only."
python -u "$SCRIPT_DIR/02_create_vimedcss_run_manifests_and_archives.py" \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --drive_root "$DRIVE_ROOT" \
  --work_root "$WORK_ROOT" \
  --make_run0 \
  --make_runA \
  --make_runB \
  --make_runC \
  --skip_runD \
  --skip_forgetting_eval \
  --manifests_only


[INFO] Cleaning previous run manifests, keeping raw manifests.
[INFO] Running Phase 2 manifests_only.
[INFO] Audio resolved: 0/15818 samples
{
  "status": "DONE",
  "subsets": 11,
  "subset_index": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/subset_index.csv",
  "report": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md"
}


In [ ]:
%%bash
set -euo pipefail

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"

echo "=== Run manifest line counts ==="
wc -l "$VIMEDCSS_ROOT/manifests/run0/"*.jsonl
wc -l "$VIMEDCSS_ROOT/manifests/runA/"*.jsonl
wc -l "$VIMEDCSS_ROOT/manifests/runB/"*.jsonl
wc -l "$VIMEDCSS_ROOT/manifests/runC/"*.jsonl

echo
echo "=== SUBSET_MATERIALIZATION_REPORT.md ==="
cat "$VIMEDCSS_ROOT/preprocessing/SUBSET_MATERIALIZATION_REPORT.md"


=== Run manifest line counts ===
   200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/run0/run0_eval_hard_200.jsonl
   200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/run0/run0_eval_test_200.jsonl
   200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/run0/run0_eval_val_200.jsonl
   600 total
   100 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/runA/runA_eval_val_100.jsonl
   200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/runA/runA_train_200.jsonl
   300 total
    200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/runB/runB_eval_hard_200.jsonl
    200 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/runB/runB_eval_val_200.jsonl
   1000 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests

### Phase 2B — Check which audio files are required

Cell này đọc các run manifests vừa tạo và tính danh sách `.wav` cần có để tạo archive thật. Nó không tải lại nếu file đã tồn tại.


In [ ]:
import csv
import json
import os
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"
PREP_DIR = VIMEDCSS_ROOT / "preprocessing"

run_manifest_paths = []
for run_dir in ["run0", "runA", "runB", "runC", "forgetting_eval"]:
    d = VIMEDCSS_ROOT / "manifests" / run_dir
    if d.exists():
        run_manifest_paths.extend(sorted(d.glob("*.jsonl")))

required = []
seen = set()

for mp in run_manifest_paths:
    with mp.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            audio = row.get("audio") or row.get("clean_audio_path") or row.get("audio_path")
            if not audio:
                continue
            basename = os.path.basename(str(audio))
            key = (row.get("source_split", "unknown"), basename)
            if key in seen:
                continue
            seen.add(key)
            local_path = AUDIO_DIR / basename
            required.append({
                "source_split": row.get("source_split", "unknown"),
                "basename": basename,
                "local_path": str(local_path),
                "exists_local": local_path.exists(),
                "source_manifest": str(mp.relative_to(VIMEDCSS_ROOT)),
            })

PREP_DIR.mkdir(parents=True, exist_ok=True)
required_csv = PREP_DIR / "required_audio_files_phase2.csv"
missing_csv = PREP_DIR / "missing_audio_files_phase2.csv"

with required_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["source_split", "basename", "local_path", "exists_local", "source_manifest"])
    writer.writeheader()
    writer.writerows(required)

missing = [r for r in required if not r["exists_local"]]
with missing_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["source_split", "basename", "local_path", "exists_local", "source_manifest"])
    writer.writeheader()
    writer.writerows(missing)

print("Required audio:", len(required))
print("Already local:", len(required) - len(missing))
print("Missing:", len(missing))
print("required_csv:", required_csv)
print("missing_csv:", missing_csv)

by_split = defaultdict(lambda: [0, 0])
for r in required:
    by_split[r["source_split"]][0] += 1
    if not r["exists_local"]:
        by_split[r["source_split"]][1] += 1

print("\nBy split: total | missing")
for split, (total, miss) in sorted(by_split.items()):
    print(f"{split:12s} {total:6d} | {miss:6d}")


Required audio: 4157
Already local: 0
Missing: 4157
required_csv: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/required_audio_files_phase2.csv
missing_csv: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/missing_audio_files_phase2.csv

By split: total | missing
hard            436 |    436
test            200 |    200
train          2918 |   2918
validation      603 |    603


### Phase 2C — Download only missing audio needed by generated subsets

Cell này **không tải full dataset**. Nó chỉ tải các file audio đang thiếu trong:

```text
preprocessing/missing_audio_files_phase2.csv
```

Nếu `Missing = 0`, cell sẽ không tải gì.


In [ ]:
import csv
import os
from collections import defaultdict
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"
MISSING_CSV = VIMEDCSS_ROOT / "preprocessing/missing_audio_files_phase2.csv"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

missing_by_split = defaultdict(set)

if MISSING_CSV.exists():
    with MISSING_CSV.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            split = row["source_split"]
            basename = row["basename"]
            if split in {"train", "validation", "test", "hard"}:
                missing_by_split[split].add(basename)

total_missing = sum(len(v) for v in missing_by_split.values())
print("Total missing audio to download:", total_missing)

if total_missing == 0:
    print("[OK] No missing audio. Skipping download.")
else:
    token = os.environ.get("HF_TOKEN") or None

    for split, needed in missing_by_split.items():
        print(f"\n[INFO] Downloading missing audio for split={split}, needed={len(needed)}")
        ds = load_dataset("tensorxt/ViMedCSS", split=split, streaming=True, token=token)

        found = 0
        for item in tqdm(ds, desc=f"scan {split}"):
            audio = item.get("audio") or {}
            path = audio.get("path") if isinstance(audio, dict) else None
            if not path:
                continue

            basename = os.path.basename(path)
            if basename not in needed:
                continue

            out_path = AUDIO_DIR / basename
            if not out_path.exists():
                audio_bytes = audio.get("bytes")
                if audio_bytes is None:
                    raise RuntimeError(f"Audio bytes missing for {split}/{basename}")
                with out_path.open("wb") as f:
                    f.write(audio_bytes)

            needed.remove(basename)
            found += 1

            if not needed:
                break

        print(f"[INFO] Found/downloaded for split={split}: {found}")
        if needed:
            print(f"[WARN] Still missing {len(needed)} files for split={split}. Examples:")
            print(sorted(list(needed))[:20])


Total missing audio to download: 4157

[INFO] Downloading missing audio for split=hard, needed=436


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

scan hard: 0it [00:00, ?it/s]

[INFO] Found/downloaded for split=hard: 436

[INFO] Downloading missing audio for split=test, needed=200


...

[INFO] Found/downloaded for split=validation: 603

[INFO] Downloading missing audio for split=train, needed=2918


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

scan train: 0it [00:00, ?it/s]

[INFO] Found/downloaded for split=train: 2918


In [ ]:
# Re-check missing audio after download.

import csv
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"
REQUIRED_CSV = VIMEDCSS_ROOT / "preprocessing/required_audio_files_phase2.csv"
MISSING_CSV = VIMEDCSS_ROOT / "preprocessing/missing_audio_files_phase2.csv"

rows = []
with REQUIRED_CSV.open("r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        row["exists_local"] = str(Path(row["local_path"]).exists())
        rows.append(row)

missing = [r for r in rows if r["exists_local"] != "True"]

with MISSING_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["source_split", "basename", "local_path", "exists_local", "source_manifest"])
    writer.writeheader()
    writer.writerows(missing)

print("Required audio:", len(rows))
print("Missing after download:", len(missing))

if missing:
    print("Examples:")
    for r in missing[:20]:
        print(r)
else:
    print("[OK] All required audio files are available locally.")


Required audio: 4157
Missing after download: 0
[OK] All required audio files are available locally.


### Phase 2D — Align raw manifests to local audio paths for materialization

Lý do cần cell này: script Phase 2 tạo archive thật sẽ copy audio từ field `audio`. Nếu raw manifest chỉ ghi `Med_CS-xxxx.wav`, script có thể không tìm được file trong `data/vimedcss_audio`.

Cell này không đổi split/selection/eval policy. Nó chỉ đổi `audio` sang absolute path local **khi file local tồn tại**, đồng thời giữ lại `audio_original` và `audio_basename`.


In [ ]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
MANIFEST_DIR = VIMEDCSS_ROOT / "manifests/raw"
AUDIO_DIR = PROJECT_ROOT / "data/vimedcss_audio"

for split in ["train", "validation", "test", "hard"]:
    p = MANIFEST_DIR / f"vimedcss_{split}_raw.jsonl"
    if not p.exists():
        print(f"[WARN] Missing raw manifest: {p}")
        continue

    df = pd.read_json(p, lines=True)

    if "audio" not in df.columns:
        if "audio_path" in df.columns:
            df["audio"] = df["audio_path"]
        elif "clean_audio_path" in df.columns:
            df["audio"] = df["clean_audio_path"]
        else:
            raise RuntimeError(f"No audio/audio_path/clean_audio_path column in {p}")

    if "audio_original" not in df.columns:
        df["audio_original"] = df["audio"]

    df["audio_basename"] = df["audio"].apply(lambda x: os.path.basename(str(x)))
    df["clean_audio_path"] = df["audio_basename"].apply(lambda x: str(AUDIO_DIR / x))

    exists_mask = df["clean_audio_path"].apply(lambda x: Path(x).exists())
    matched = int(exists_mask.sum())

    # Chỉ những dòng có local audio mới được đổi sang absolute path.
    df.loc[exists_mask, "audio"] = df.loc[exists_mask, "clean_audio_path"]

    df.to_json(p, orient="records", lines=True, force_ascii=False)
    print(f"[OK] {split}: local audio matched {matched}/{len(df)} -> {p}")


/tmp/ipykernel_2501/1225507657.py:38: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


[OK] train: local audio matched 2918/11832 -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_train_raw.jsonl
[OK] validation: local audio matched 603/1714 -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_validation_raw.jsonl


/tmp/ipykernel_2501/1225507657.py:38: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)
/tmp/ipykernel_2501/1225507657.py:38: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


[OK] test: local audio matched 200/1614 -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_test_raw.jsonl
[OK] hard: local audio matched 436/658 -> /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/manifests/raw/vimedcss_hard_raw.jsonl


/tmp/ipykernel_2501/1225507657.py:38: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df.to_json(p, orient="records", lines=True, force_ascii=False)


### Phase 2E — Remove old broken archives

Chạy cell này trước khi tạo archive thật để tránh giữ lại các file `0.00 MB`.


In [ ]:
%%bash
set -euo pipefail

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"

mkdir -p "$VIMEDCSS_ROOT/preprocessing/subset_archives"
mkdir -p "$DRIVE_ROOT/subset_archives"

echo "[INFO] Removing old local archives."
rm -f "$VIMEDCSS_ROOT/preprocessing/subset_archives/"*.tar.gz

echo "[INFO] Removing old Drive archives."
rm -f "$DRIVE_ROOT/subset_archives/"*.tar.gz

echo "[DONE]"


[INFO] Removing old local archives.
[INFO] Removing old Drive archives.
[DONE]


### Phase 2F — Test materialization with Run A first

Chạy Run A trước vì nhẹ. Cell này **không có** `--manifests_only`.


In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WORK_ROOT="/content/vimedcss_work"

cd "$PROJECT_ROOT"
mkdir -p "$WORK_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

python "$SCRIPT_DIR/02_create_vimedcss_run_manifests_and_archives.py" \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --drive_root "$DRIVE_ROOT" \
  --work_root "$WORK_ROOT" \
  --make_runA \
  --skip_runD \
  --skip_forgetting_eval


[INFO] Audio resolved: 4157/15818 samples
{
  "status": "DONE",
  "subsets": 2,
  "subset_index": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/subset_index.csv",
  "report": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md"
}


In [ ]:
%%bash
set -euo pipefail

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"

echo "=== Local Run A archives ==="
ls -lh "$VIMEDCSS_ROOT/preprocessing/subset_archives" | grep runA || true

echo
echo "=== Drive Run A archives ==="
ls -lh "$DRIVE_ROOT/subset_archives" | grep runA || true

echo
echo "=== Validate runA_train_200.tar.gz ==="
TAR="$VIMEDCSS_ROOT/preprocessing/subset_archives/runA_train_200.tar.gz"
ls -lh "$TAR"
echo "files inside: $(tar -tzf "$TAR" | wc -l)"
echo "audio files inside: $(tar -tzf "$TAR" | grep -E '\.(wav|flac|mp3)$' | wc -l)"
tar -xOzf "$TAR" runA_train_200/subset_metadata.json | python -m json.tool


=== Local Run A archives ===
-rw-r--r-- 1 root root  87M Jun 15 19:01 runA_eval_val_100.tar.gz
-rw-r--r-- 1 root root 183M Jun 15 19:01 runA_train_200.tar.gz

=== Drive Run A archives ===
-rw------- 1 root root  87M Jun 15 19:01 runA_eval_val_100.tar.gz
-rw------- 1 root root 183M Jun 15 19:01 runA_train_200.tar.gz

=== Validate runA_train_200.tar.gz ===
-rw-r--r-- 1 root root 183M Jun 15 19:01 /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/subset_archives/runA_train_200.tar.gz
files inside: 204
audio files inside: 200
{
    "subset_name": "runA_train_200",
    "rows": 200,
    "total_hours": 0.4294,
    "copied_audio": 200,
    "missing_audio": 0,
    "created_at": "2026-06-15T19:00:52.152809+00:00"
}


### Phase 2G — Materialize all Phase 2 archives

Chỉ chạy cell này sau khi Run A archive có audio count đúng. Cell này **không có** `--manifests_only`.


In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
VIMEDCSS_ROOT="$PROJECT_ROOT/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WORK_ROOT="/content/vimedcss_work"

cd "$PROJECT_ROOT"
mkdir -p "$WORK_ROOT"

SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"
if [ ! -d "$SCRIPT_DIR" ]; then
  SCRIPT_DIR="$PROJECT_ROOT/scripts/vimedcss"
fi

python "$SCRIPT_DIR/02_create_vimedcss_run_manifests_and_archives.py" \
  --vimedcss_root "$VIMEDCSS_ROOT" \
  --drive_root "$DRIVE_ROOT" \
  --work_root "$WORK_ROOT" \
  --make_run0 \
  --make_runA \
  --make_runB \
  --make_runC \
  --skip_runD \
  --skip_forgetting_eval


[INFO] Audio resolved: 4157/15818 samples
{
  "status": "DONE",
  "subsets": 11,
  "subset_index": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/subset_index.csv",
  "report": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md"
}


In [ ]:
%%bash
set -euo pipefail

VIMEDCSS_ROOT="/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"

echo "=== Final local subset archives ==="
ls -lh "$VIMEDCSS_ROOT/preprocessing/subset_archives"

echo
echo "=== Final Drive subset archives ==="
ls -lh "$DRIVE_ROOT/subset_archives"

echo
echo "=== Audio count per local archive ==="
for tar in "$VIMEDCSS_ROOT"/preprocessing/subset_archives/*.tar.gz; do
  name=$(basename "$tar")
  size=$(du -h "$tar" | cut -f1)
  audio_count=$(tar -tzf "$tar" | grep -E '\.(wav|flac|mp3)$' | wc -l)
  echo "$name | size=$size | audio_files=$audio_count"
done

echo
echo "=== Final materialization report ==="
cat "$VIMEDCSS_ROOT/preprocessing/SUBSET_MATERIALIZATION_REPORT.md"


=== Final local subset archives ===
total 3.9G
-rw-r--r-- 1 root root 172M Jun 15 19:09 run0_eval_hard_200.tar.gz
-rw-r--r-- 1 root root 171M Jun 15 19:09 run0_eval_test_200.tar.gz
-rw-r--r-- 1 root root 163M Jun 15 19:08 run0_eval_val_200.tar.gz
-rw-r--r-- 1 root root  87M Jun 15 19:10 runA_eval_val_100.tar.gz
-rw-r--r-- 1 root root 183M Jun 15 19:10 runA_train_200.tar.gz
-rw-r--r-- 1 root root 167M Jun 15 19:13 runB_eval_hard_200.tar.gz
-rw-r--r-- 1 root root 172M Jun 15 19:12 runB_eval_val_200.tar.gz
-rw-r--r-- 1 root root 864M Jun 15 19:12 runB_train_1000.tar.gz
-rw-r--r-- 1 root root 171M Jun 15 19:17 runC_eval_hard_200.tar.gz
-rw-r--r-- 1 root root 160M Jun 15 19:17 runC_eval_val_200.tar.gz
-rw-r--r-- 1 root root 1.6G Jun 15 19:16 runC_train_4h.tar.gz

=== Final Drive subset archives ===
total 3.9G
-rw------- 1 root root 172M Jun 15 19:09 run0_eval_hard_200.tar.gz
-rw------- 1 root root 171M Jun 15 19:09 run0_eval_test_200.tar.gz
-rw------- 1 root root 163M Jun 15 19:08 run0_eval

In [ ]:
# Fail-fast validation: archive train/eval subsets must contain audio.
# This prevents accidentally moving to Phase 3 with 0 MB archives.

import tarfile
from pathlib import Path

VIMEDCSS_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/vimedcss")
archive_dir = VIMEDCSS_ROOT / "preprocessing/subset_archives"

bad = []

for tar_path in sorted(archive_dir.glob("*.tar.gz")):
    with tarfile.open(tar_path, "r:gz") as tar:
        names = tar.getnames()
    audio_count = sum(1 for n in names if n.lower().endswith((".wav", ".flac", ".mp3")))
    size_mb = tar_path.stat().st_size / (1024 * 1024)
    print(f"{tar_path.name:35s} | {size_mb:8.2f} MB | audio={audio_count}")
    if audio_count == 0:
        bad.append(tar_path.name)

if bad:
    raise RuntimeError(f"Bad archives with zero audio files: {bad}")

print("[OK] All archives contain audio files.")


run0_eval_hard_200.tar.gz           |   171.68 MB | audio=200
run0_eval_test_200.tar.gz           |   170.63 MB | audio=200
run0_eval_val_200.tar.gz            |   162.63 MB | audio=200
runA_eval_val_100.tar.gz            |    86.16 MB | audio=100
runA_train_200.tar.gz               |   182.58 MB | audio=200
runB_eval_hard_200.tar.gz           |   166.86 MB | audio=200
runB_eval_val_200.tar.gz            |   171.65 MB | audio=200
runB_train_1000.tar.gz              |   863.53 MB | audio=1000
runC_eval_hard_200.tar.gz           |   170.49 MB | audio=200
runC_eval_val_200.tar.gz            |   159.56 MB | audio=200
runC_train_4h.tar.gz                |  1633.68 MB | audio=1938
[OK] All archives contain audio files.


### Phase 2H — Bundle small Phase 2 artifacts for local download


In [ ]:
# Bundle Phase 2 run manifests + reports/index.
# Không bundle subset_archives vì các file .tar.gz audio có thể lớn và đã được copy sang Drive.

import tarfile
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
OUT = Path("/content/phase2_vimedcss_manifest_artifacts.tar.gz")

items = [
    VIMEDCSS_ROOT / "manifests/run0",
    VIMEDCSS_ROOT / "manifests/runA",
    VIMEDCSS_ROOT / "manifests/runB",
    VIMEDCSS_ROOT / "manifests/runC",
    VIMEDCSS_ROOT / "manifests/forgetting_eval",
    VIMEDCSS_ROOT / "preprocessing/subset_index.csv",
    VIMEDCSS_ROOT / "preprocessing/SUBSET_MATERIALIZATION_REPORT.md",
    VIMEDCSS_ROOT / "preprocessing/subset_materialization_summary.json",
    VIMEDCSS_ROOT / "preprocessing/required_audio_files_phase2.csv",
    VIMEDCSS_ROOT / "preprocessing/missing_audio_files_phase2.csv",
]

with tarfile.open(OUT, "w:gz") as tar:
    for item in items:
        if item.exists():
            tar.add(item, arcname=str(item.relative_to(PROJECT_ROOT)))
            print("[ADD]", item.relative_to(PROJECT_ROOT))
        else:
            print("[SKIP missing]", item)

print("Created:", OUT)


[ADD] experiments/asr/vimedcss/manifests/run0
[ADD] experiments/asr/vimedcss/manifests/runA
[ADD] experiments/asr/vimedcss/manifests/runB
[ADD] experiments/asr/vimedcss/manifests/runC
[ADD] experiments/asr/vimedcss/manifests/forgetting_eval
[ADD] experiments/asr/vimedcss/preprocessing/subset_index.csv
[ADD] experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md
[ADD] experiments/asr/vimedcss/preprocessing/subset_materialization_summary.json
[ADD] experiments/asr/vimedcss/preprocessing/required_audio_files_phase2.csv
[ADD] experiments/asr/vimedcss/preprocessing/missing_audio_files_phase2.csv
Created: /content/phase2_vimedcss_manifest_artifacts.tar.gz


In [ ]:
%%bash
set -euo pipefail

DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
mkdir -p "$DRIVE_ROOT/artifact_bundles"

cp -v /content/week6_phase0_light_artifacts.tar.gz "$DRIVE_ROOT/artifact_bundles/" || true
cp -v /content/phase1_vimedcss_data_audit_artifacts.tar.gz "$DRIVE_ROOT/artifact_bundles/" || true
cp -v /content/phase2_vimedcss_manifest_artifacts.tar.gz "$DRIVE_ROOT/artifact_bundles/" || true

echo
echo "=== Drive artifact bundles ==="
ls -lh "$DRIVE_ROOT/artifact_bundles"


'/content/week6_phase0_light_artifacts.tar.gz' -> '/content/drive/MyDrive/clinical_asr_vimedcss/artifact_bundles/week6_phase0_light_artifacts.tar.gz'
'/content/phase1_vimedcss_data_audit_artifacts.tar.gz' -> '/content/drive/MyDrive/clinical_asr_vimedcss/artifact_bundles/phase1_vimedcss_data_audit_artifacts.tar.gz'
'/content/phase2_vimedcss_manifest_artifacts.tar.gz' -> '/content/drive/MyDrive/clinical_asr_vimedcss/artifact_bundles/phase2_vimedcss_manifest_artifacts.tar.gz'

=== Drive artifact bundles ===
total 1.9M
-rw------- 1 root root 1.1M Jun 15 19:20 phase1_vimedcss_data_audit_artifacts.tar.gz
-rw------- 1 root root 804K Jun 15 19:20 phase2_vimedcss_manifest_artifacts.tar.gz
-rw------- 1 root root  15K Jun 15 19:20 week6_phase0_light_artifacts.tar.gz


### Tải về các tệp Manifest và Báo cáo Phase 2

Cell này sẽ nén các tệp cấu hình run và báo cáo thành một file ZIP duy nhất để bạn lưu trữ tại máy cá nhân.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
VIMEDCSS_ROOT = PROJECT_ROOT / "experiments/asr/vimedcss"
ZIP_OUT = "/content/ViMedCSS_Phase2_Manifests_And_Reports.zip"

# Danh sách các thư mục và tệp cần nén
target_items = [
    "experiments/asr/vimedcss/manifests/run0/",
    "experiments/asr/vimedcss/manifests/runA/",
    "experiments/asr/vimedcss/manifests/runB/",
    "experiments/asr/vimedcss/manifests/runC/",
    "experiments/asr/vimedcss/manifests/forgetting_eval/",
    "experiments/asr/vimedcss/preprocessing/subset_index.csv",
    "experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md",
    "experiments/asr/vimedcss/preprocessing/subset_materialization_summary.json"
]

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for item_path in target_items:
        full_path = PROJECT_ROOT / item_path

        if full_path.exists():
            if full_path.is_dir():
                for root, dirs, filenames in os.walk(full_path):
                    for filename in filenames:
                        file_full_path = Path(root) / filename
                        arcname = file_full_path.relative_to(PROJECT_ROOT)
                        zipf.write(file_full_path, arcname=arcname)
                        print(f"[ADD] {arcname}")
            else:
                arcname = full_path.relative_to(PROJECT_ROOT)
                zipf.write(full_path, arcname=arcname)
                print(f"[ADD] {arcname}")
        else:
            print(f"[SKIP missing] {item_path}")

print(f"\nCreated: {ZIP_OUT}")
try:
    files.download(ZIP_OUT)
except Exception as e:
    print(f"Lỗi khi tải file: {e}")

[ADD] experiments/asr/vimedcss/manifests/run0/run0_eval_hard_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/run0/run0_eval_test_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/run0/run0_eval_val_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runA/runA_eval_val_100.jsonl
[ADD] experiments/asr/vimedcss/manifests/runA/runA_train_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runB/runB_eval_hard_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runB/runB_train_1000.jsonl
[ADD] experiments/asr/vimedcss/manifests/runB/runB_eval_val_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runC/runC_eval_val_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runC/runC_eval_hard_200.jsonl
[ADD] experiments/asr/vimedcss/manifests/runC/runC_train_4h.jsonl
[ADD] experiments/asr/vimedcss/manifests/forgetting_eval/.gitkeep
[ADD] experiments/asr/vimedcss/preprocessing/subset_index.csv
[ADD] experiments/asr/vimedcss/preprocessing/SUBSET_MATERIALIZATION_REPORT.md
[ADD] experiments/asr/vimedcs

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Tải về các Audio Subset Archives (Phase 2)

Cell này sẽ nén các tệp audio `.tar.gz` lớn đã được materialize thành một file ZIP duy nhất.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")
ARCHIVE_DIR = PROJECT_ROOT / "experiments/asr/vimedcss/preprocessing/subset_archives"
ZIP_OUT_AUDIO = "/content/ViMedCSS_Phase2_Audio_Archives.zip"

# Danh sách các tệp cụ thể cần nén
target_files = [
    "run0_eval_val_200.tar.gz",
    "run0_eval_hard_200.tar.gz",
    "run0_eval_test_200.tar.gz",
    "runA_train_200.tar.gz",
    "runA_eval_val_100.tar.gz",
    "runB_train_1000.tar.gz",
    "runB_eval_val_200.tar.gz",
    "runB_eval_hard_200.tar.gz",
    "runC_train_4h.tar.gz",
    "runC_eval_val_200.tar.gz",
    "runC_eval_hard_200.tar.gz",
    "vietmed_dev_small.tar.gz"
]

print(f"Bắt đầu nén các tệp audio vào {ZIP_OUT_AUDIO}...")

with zipfile.ZipFile(ZIP_OUT_AUDIO, 'w', zipfile.ZIP_STORED) as zipf:
    for filename in target_files:
        full_path = ARCHIVE_DIR / filename
        if full_path.exists():
            # Giữ cấu trúc thư mục từ experiments/...
            arcname = full_path.relative_to(PROJECT_ROOT)
            zipf.write(full_path, arcname=arcname)
            print(f"[ADD] {arcname} ({full_path.stat().st_size / (1024*1024):.2f} MB)")
        else:
            print(f"[SKIP missing] {filename}")

print(f"\nĐã tạo xong: {ZIP_OUT_AUDIO}")
print("Đang khởi tạo quá trình tải về. Lưu ý: Dung lượng lớn có thể gây chậm trình duyệt.")

try:
    files.download(ZIP_OUT_AUDIO)
except Exception as e:
    print(f"Lỗi khi tải file: {e}")

Bắt đầu nén các tệp audio vào /content/ViMedCSS_Phase2_Audio_Archives.zip...
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/run0_eval_val_200.tar.gz (162.63 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/run0_eval_hard_200.tar.gz (171.68 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/run0_eval_test_200.tar.gz (170.63 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runA_train_200.tar.gz (182.58 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runA_eval_val_100.tar.gz (86.16 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runB_train_1000.tar.gz (863.53 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runB_eval_val_200.tar.gz (171.65 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runB_eval_hard_200.tar.gz (166.86 MB)
[ADD] experiments/asr/vimedcss/preprocessing/subset_archives/runC_train_4h.tar.gz (1633.68 MB)
[ADD] experiments/asr/vimedcss/preprocessing/su

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Phase 3.I — Run 0 baseline trên ViMedCSS (Internal ViMedCSS Run 0 baseline)

Phase 3 nối tiếp Phase 0/1/2: dùng **subset archives của Run 0** đã tạo ở Phase 2 để chạy baseline cho checkpoint Week 5 PhoWhisper VietMed trên ViMedCSS.

Mục tiêu:

```text
Run 0 baseline → Run A smoke → Run B mini → Run C balanced 4h
```

Guardrails:

```text
- Không train.
- Không chọn checkpoint.
- Không dùng test/hard để chỉnh hyperparameter.
- Validation 200 là baseline chính để so với Run A/B/C.
- Hard 200 chỉ để reporting/stress.
- Test 200 nếu chạy thì reporting-only.
```

Điều kiện bắt buộc trước khi chạy:

```text
- Phase 2 archive thật đã có audio, không phải archive 0.00 MB.
- run0_eval_val_200.tar.gz tồn tại trên Drive.
- run0_eval_hard_200.tar.gz tồn tại trên Drive.
- Week 5 checkpoint đã đầy đủ file model/tokenizer/config.
- Colab đang dùng GPU runtime nếu muốn chạy đủ 200 samples nhanh.
```


### Phase 3A — Khai báo biến và tạo thư mục output

Cell này chỉ khai báo path. Không train và không chạy inference.

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WORK_ROOT="/content/vimedcss_work"
MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"

cd "$PROJECT_ROOT"

mkdir -p "$WORK_ROOT"
mkdir -p "$DRIVE_ROOT/baseline/run0_week5_checkpoint"
mkdir -p "$DRIVE_ROOT/baseline/run0_original_phowhisper"

cat > /content/phase3_env.sh <<EOF
export PROJECT_ROOT="$PROJECT_ROOT"
export DRIVE_ROOT="$DRIVE_ROOT"
export WORK_ROOT="$WORK_ROOT"
export MODEL="$MODEL"
EOF

echo "[OK] Phase 3 environment written to /content/phase3_env.sh"
cat /content/phase3_env.sh

[OK] Phase 3 environment written to /content/phase3_env.sh
export PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
export DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
export WORK_ROOT="/content/vimedcss_work"
export MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"


### Phase 3B — Kiểm tra GPU và dependencies tối thiểu

Kỳ vọng `cuda_available: True`. Nếu đang ở CPU runtime, vẫn có thể smoke test rất ít mẫu nhưng không nên chạy full 200 samples.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python - <<'PY'
import torch
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
PY

python - <<'PY'
missing = []
for pkg in ["torch", "transformers", "soundfile", "librosa"]:
    try:
        __import__(pkg)
        print(f"[OK] {pkg}")
    except Exception as e:
        print(f"[MISSING] {pkg}: {e}")
        missing.append(pkg)
if missing:
    raise SystemExit("Missing packages. Install dependencies before Phase 3 full run.")
PY

cuda_available: True
gpu: Tesla T4
[OK] torch
[OK] transformers
[OK] soundfile
[OK] librosa


### Phase 3C — Kiểm tra Phase 2 Run 0 archives

Archive hợp lệ phải:

```text
- tồn tại trên Drive
- size không quá nhỏ
- extract được
- có file .jsonl
- có audio files bên trong
```

Nếu archive báo `0.00 MB` hoặc `audio_files=0`, quay lại Phase 2 materialization.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

ARCHIVE_DIR="$DRIVE_ROOT/subset_archives"
REQUIRED=(
  "run0_eval_val_200.tar.gz"
  "run0_eval_hard_200.tar.gz"
  "run0_eval_test_200.tar.gz"
)

for name in "${REQUIRED[@]}"; do
  tar_path="$ARCHIVE_DIR/$name"
  echo "========== $name =========="
  if [ ! -f "$tar_path" ]; then
    echo "[FAIL] missing archive: $tar_path"
    exit 1
  fi

  size_bytes=$(stat -c%s "$tar_path")
  echo "size_bytes=$size_bytes"
  if [ "$size_bytes" -le 102400 ]; then
    echo "[FAIL] archive too small; likely broken/empty: $tar_path"
    exit 1
  fi

  jsonl_count=$(tar -tzf "$tar_path" | grep -E '\.jsonl$' | wc -l)
  audio_count=$(tar -tzf "$tar_path" | grep -Ei '\.(wav|flac|mp3|ogg|m4a|aac|wma)$' | wc -l)
  echo "jsonl_count=$jsonl_count"
  echo "audio_count=$audio_count"

  if [ "$jsonl_count" -eq 0 ]; then
    echo "[FAIL] no jsonl manifest inside archive"
    exit 1
  fi
  if [ "$audio_count" -eq 0 ]; then
    echo "[FAIL] no audio files inside archive"
    exit 1
  fi

done

echo "[PASS] Run 0 archives look usable."

========== run0_eval_val_200.tar.gz ==========
size_bytes=170533763
jsonl_count=1
audio_count=200
========== run0_eval_hard_200.tar.gz ==========
size_bytes=180019366
jsonl_count=1
audio_count=200
========== run0_eval_test_200.tar.gz ==========
size_bytes=178923011
jsonl_count=1
audio_count=200
[PASS] Run 0 archives look usable.


### Phase 3D — Kiểm tra checkpoint Week 5

Warning `week5_checkpoint_missing_or_incomplete` chặn Phase 3. Checkpoint hợp lệ cần có tối thiểu config, generation config, preprocessor config, tokenizer và model weights.

#### Tải lại checkpoint từ HF repo run01

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise RuntimeError("Chưa có HF_TOKEN trong Colab Secrets.")

snapshot_download(
    repo_id="DukeShy/phowhisper-medium-medical-vi-run01",
    repo_type="model",
    local_dir="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint",
    token=HF_TOKEN,
)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

'/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint'

In [ ]:
%%bash
set -e

MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"

echo "MODEL=$MODEL"
find "$MODEL" -maxdepth 1 -type f | sort

MODEL=/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/generation_config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/.gitattributes
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/medical_errors_dev.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/metrics_dev_project_wer.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/model.safetensors
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/predictions_dev.jsonl
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/preprocessor_config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/processor_config.json
/content/drive

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
WORK_ROOT="/content/vimedcss_work"
MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"

cat > /content/phase3_env.sh <<EOF
export PROJECT_ROOT="$PROJECT_ROOT"
export DRIVE_ROOT="$DRIVE_ROOT"
export WORK_ROOT="$WORK_ROOT"
export MODEL="$MODEL"
EOF

cat /content/phase3_env.sh

export PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
export DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
export WORK_ROOT="/content/vimedcss_work"
export MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"


In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

echo "MODEL=$MODEL"
if [ ! -d "$MODEL" ]; then
  echo "[FAIL] checkpoint directory does not exist: $MODEL"
  exit 1
fi

find "$MODEL" -maxdepth 1 -type f | sort

python - <<'PY'
import os
from pathlib import Path
model = Path(os.environ.get("MODEL", "/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"))
files = {p.name for p in model.glob("*") if p.is_file()}
required_exact = ["config.json", "generation_config.json", "preprocessor_config.json"]
missing = [x for x in required_exact if x not in files]
if not ({"tokenizer.json", "tokenizer_config.json"} & files):
    missing.append("tokenizer.json or tokenizer_config.json")
if not ({"model.safetensors", "pytorch_model.bin"} & files):
    missing.append("model.safetensors or pytorch_model.bin")
if missing:
    print("[FAIL] incomplete checkpoint. Missing:")
    for x in missing:
        print(" -", x)
    raise SystemExit(1)
print("[PASS] checkpoint looks complete")
PY

MODEL=/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/generation_config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/.gitattributes
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/medical_errors_dev.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/metrics_dev_project_wer.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/model.safetensors
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/predictions_dev.jsonl
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/preprocessor_config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/processor_config.json
/content/drive

### Phase 3E — Tạo script eval Run 0

Script này extract archive, tìm manifest, resolve audio path, chạy PhoWhisper inference, tính WER/CER và xuất predictions/metrics. Không train.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

SCRIPT_DIR="scripts/asr_vimedcss"
mkdir -p "$SCRIPT_DIR"
SCRIPT="$SCRIPT_DIR/03_run_phowhisper_eval_on_archive.py"

cat > "$SCRIPT" <<'PYCODE'
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
from __future__ import annotations

import argparse, json, math, re, shutil, tarfile, time, unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import torch
from transformers import pipeline

TEXT_KEYS = ["segment_text", "sentence", "reference_text", "transcript_text", "text"]
AUDIO_KEYS = ["audio", "audio_path", "path", "wav_path", "file", "file_path"]
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".wma"}

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                row = json.loads(line)
                row["_line_no"] = line_no
                rows.append(row)
    return rows

def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def safe_extract_tar(tar_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_abs = dest_dir.resolve()
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            target = (dest_dir / member.name).resolve()
            if not str(target).startswith(str(dest_abs)):
                raise RuntimeError(f"Unsafe tar path: {member.name}")
        tar.extractall(dest_dir)

def find_manifest(extract_dir: Path, explicit_manifest: str | None = None) -> Path:
    if explicit_manifest:
        p = Path(explicit_manifest)
        q = p if p.is_absolute() else extract_dir / explicit_manifest
        if q.exists():
            return q
        raise FileNotFoundError(f"Explicit manifest not found: {explicit_manifest}")
    files = sorted(extract_dir.rglob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No JSONL manifest found under {extract_dir}")
    preferred = [p for p in files if any(k in p.name.lower() for k in ["manifest", "run0", "eval", "vimedcss"])]
    return preferred[0] if preferred else files[0]

def pick_reference(row: dict[str, Any]) -> tuple[str, str | None]:
    for key in TEXT_KEYS:
        v = row.get(key)
        if isinstance(v, str) and v.strip():
            return v.strip(), key
    return "", None

def audio_value_to_path_like(v: Any) -> str | None:
    if v is None:
        return None
    if isinstance(v, str):
        return v.strip() or None
    if isinstance(v, dict):
        for k in ["path", "file", "filename"]:
            if v.get(k):
                return str(v[k]).strip()
    return None

def build_basename_index(extract_dir: Path) -> dict[str, Path]:
    return {p.name: p for p in extract_dir.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS}

def resolve_audio_path(row: dict[str, Any], extract_dir: Path, manifest_dir: Path, basename_index: dict[str, Path]):
    for key in AUDIO_KEYS:
        raw = audio_value_to_path_like(row.get(key))
        if not raw:
            continue
        p = Path(raw)
        candidates = [p] if p.is_absolute() else [manifest_dir / p, extract_dir / p, extract_dir / p.name]
        for c in candidates:
            if c.exists() and c.is_file():
                return c, key, raw
        if p.name in basename_index:
            return basename_index[p.name], key, raw
    return None, None, None

def ws(text: str) -> str:
    return " ".join(text.strip().split())

def norm(text: str) -> str:
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\sÀ-ỹà-ỹ]", " ", text, flags=re.UNICODE).replace("_", " ")
    return ws(text)

def edit_distance(a: list[Any], b: list[Any]) -> int:
    prev = list(range(len(b) + 1))
    cur = [0] * (len(b) + 1)
    for i in range(1, len(a) + 1):
        cur[0] = i
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev, cur = cur, prev
    return prev[len(b)]

def compute_wer(refs: list[str], hyps: list[str], normalized: bool = False):
    edits, total = 0, 0
    for ref, hyp in zip(refs, hyps):
        ref = norm(ref) if normalized else ws(ref)
        hyp = norm(hyp) if normalized else ws(hyp)
        r, h = ref.split(), hyp.split()
        edits += edit_distance(r, h)
        total += len(r)
    return None if total == 0 else edits / total

def compute_cer(refs: list[str], hyps: list[str], normalized: bool = False):
    edits, total = 0, 0
    for ref, hyp in zip(refs, hyps):
        ref = norm(ref) if normalized else ws(ref)
        hyp = norm(hyp) if normalized else ws(hyp)
        edits += edit_distance(list(ref), list(hyp))
        total += len(ref)
    return None if total == 0 else edits / total

def to_float(v: Any):
    try:
        if v is None or isinstance(v, bool):
            return None
        x = float(v)
        return None if math.isnan(x) or math.isinf(x) else x
    except Exception:
        return None

def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--archive", required=True)
    ap.add_argument("--work_dir", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--output_predictions", required=True)
    ap.add_argument("--output_metrics", required=True)
    ap.add_argument("--manifest_inside_archive", default=None)
    ap.add_argument("--language", default="vi")
    ap.add_argument("--task", default="transcribe")
    ap.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    ap.add_argument("--max_samples", type=int, default=None)
    ap.add_argument("--clean_work_dir", action="store_true")
    ap.add_argument("--eval_name", default=None)
    ap.add_argument("--model_label", default=None)
    args = ap.parse_args()

    archive = Path(args.archive)
    work_dir = Path(args.work_dir)
    out_pred = Path(args.output_predictions)
    out_metrics = Path(args.output_metrics)
    if not archive.exists():
        raise FileNotFoundError(f"Archive not found: {archive}")
    if args.clean_work_dir and work_dir.exists():
        shutil.rmtree(work_dir)
    extract_dir = work_dir / "extracted"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    print(f"[INFO] Extracting: {archive}")
    safe_extract_tar(archive, extract_dir)
    manifest = find_manifest(extract_dir, args.manifest_inside_archive)
    print(f"[INFO] Manifest: {manifest}")

    rows = read_jsonl(manifest)
    if args.max_samples is not None:
        rows = rows[:args.max_samples]
    basename_index = build_basename_index(extract_dir)

    prepared, skipped = [], []
    for row in rows:
        ref, ref_key = pick_reference(row)
        audio, audio_key, audio_raw = resolve_audio_path(row, extract_dir, manifest.parent, basename_index)
        sid = row.get("segment_id") or row.get("sample_id") or f"line_{row.get('_line_no')}"
        reasons = []
        if not ref:
            reasons.append("missing_reference_text")
        if audio is None:
            reasons.append("audio_not_resolved")
        if reasons:
            skipped.append({"segment_id": sid, "line_no": row.get("_line_no"), "reasons": reasons, "audio_key": audio_key, "audio_raw": audio_raw, "ref_key": ref_key})
        else:
            prepared.append({"segment_id": sid, "row": row, "audio": audio, "reference_text": ref, "ref_key": ref_key, "audio_key": audio_key, "audio_raw": audio_raw})
    if not prepared:
        raise RuntimeError("No valid samples prepared for evaluation.")

    if args.device == "auto":
        device = 0 if torch.cuda.is_available() else -1
    elif args.device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable.")
        device = 0
    else:
        device = -1
    model_label = args.model_label or str(args.model)
    eval_name = args.eval_name or archive.stem.replace(".tar", "")

    print(f"[INFO] Loading ASR pipeline: {args.model}")
    asr = pipeline("automatic-speech-recognition", model=args.model, tokenizer=args.model, feature_extractor=args.model, device=device)

    preds, refs, hyps = [], [], []
    start = time.time()
    for i, item in enumerate(prepared, start=1):
        sample_t0 = time.time()
        err = None
        try:
            out = asr(str(item["audio"]), generate_kwargs={"language": args.language, "task": args.task})
            hyp = out.get("text", "") if isinstance(out, dict) else str(out)
        except Exception as exc:
            hyp = ""
            err = str(exc)
        runtime = time.time() - sample_t0
        row = item["row"]
        ref = item["reference_text"]
        refs.append(ref)
        hyps.append(hyp)
        preds.append({
            "segment_id": item["segment_id"],
            "sample_id": row.get("sample_id") or item["segment_id"],
            "eval_name": eval_name,
            "model_label": model_label,
            "model": str(args.model),
            "archive": str(archive),
            "audio": str(item["audio"]),
            "reference_text": ref,
            "prediction_text": hyp,
            "runtime_seconds": round(runtime, 4),
            "error": err,
            "duration_seconds": row.get("duration_seconds"),
            "topic": row.get("topic"),
            "cs_terms_count": row.get("cs_terms_count"),
            "cs_terms_list": row.get("cs_terms_list"),
            "original_video_link": row.get("original_video_link"),
            "original_video_title": row.get("original_video_title"),
            "start_time": row.get("start_time"),
            "end_time": row.get("end_time"),
        })
        if i % 10 == 0 or i == len(prepared):
            print(f"[INFO] processed {i}/{len(prepared)}")

    total_runtime = time.time() - start
    durations = [to_float(p.get("duration_seconds")) for p in preds]
    durations = [d for d in durations if d is not None]
    metrics = {
        "created_at": now_iso(),
        "phase": "phase3_run0_baseline",
        "eval_name": eval_name,
        "model": str(args.model),
        "model_label": model_label,
        "archive": str(archive),
        "manifest": str(manifest),
        "n_manifest_rows": len(rows),
        "n_prepared": len(prepared),
        "n_skipped": len(skipped),
        "skipped_preview": skipped[:100],
        "strict_wer": compute_wer(refs, hyps, normalized=False),
        "strict_cer": compute_cer(refs, hyps, normalized=False),
        "normalized_wer": compute_wer(refs, hyps, normalized=True),
        "normalized_cer": compute_cer(refs, hyps, normalized=True),
        "total_runtime_seconds": round(total_runtime, 4),
        "runtime_seconds_per_sample": round(total_runtime / len(prepared), 4),
        "total_audio_hours": round(sum(durations) / 3600, 4) if durations else None,
        "language": args.language,
        "task": args.task,
        "effective_device": device,
        "max_samples": args.max_samples,
        "does_not_train": True,
    }
    write_jsonl(out_pred, preds)
    write_json(out_metrics, metrics)
    print(json.dumps({"status": "DONE", "eval_name": eval_name, "n_prepared": len(prepared), "n_skipped": len(skipped), "strict_wer": metrics["strict_wer"], "strict_cer": metrics["strict_cer"], "normalized_wer": metrics["normalized_wer"], "normalized_cer": metrics["normalized_cer"], "predictions": str(out_pred), "metrics": str(out_metrics)}, ensure_ascii=False, indent=2))

if __name__ == "__main__":
    main()
PYCODE

chmod +x "$SCRIPT"
echo "[OK] Created $SCRIPT"
python "$SCRIPT" --help | head -40

[OK] Created scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py
usage: 03_run_phowhisper_eval_on_archive.py [-h] --archive ARCHIVE --work_dir
                                            WORK_DIR --model MODEL
                                            --output_predictions
                                            OUTPUT_PREDICTIONS
                                            --output_metrics OUTPUT_METRICS
                                            [--manifest_inside_archive MANIFEST_INSIDE_ARCHIVE]
                                            [--language LANGUAGE]
                                            [--task TASK]
                                            [--device {auto,cpu,cuda}]
                                            [--max_samples MAX_SAMPLES]
                                            [--clean_work_dir]
                                            [--eval_name EVAL_NAME]
                                            [--model_label MODEL_LABEL]

options:
  -h

### Phase 3F — Smoke test 3 samples trên validation

Chỉ chạy 3 mẫu để xác nhận checkpoint load được, archive extract được, audio resolve được và prediction không rỗng hàng loạt.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_val_200.tar.gz" \
  --work_dir "$WORK_ROOT/smoke_baseline_val" \
  --model "$MODEL" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_week5_checkpoint/smoke_predictions_val_3.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_week5_checkpoint/smoke_metrics_val_3.json" \
  --max_samples 3 \
  --clean_work_dir \
  --eval_name "smoke_val_3" \
  --model_label "week5_vietmed_checkpoint"

[INFO] Extracting: /content/drive/MyDrive/clinical_asr_vimedcss/subset_archives/run0_eval_val_200.tar.gz
[INFO] Manifest: /content/vimedcss_work/smoke_baseline_val/extracted/run0_eval_val_200/manifest.jsonl
[INFO] Loading ASR pipeline: /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
[INFO] processed 3/3
{
  "status": "DONE",
  "eval_name": "smoke_val_3",
  "n_prepared": 3,
  "n_skipped": 0,
  "strict_wer": 0.2518518518518518,
  "strict_cer": 0.18363939899833054,
  "normalized_wer": 0.17647058823529413,
  "normalized_cer": 0.16129032258064516,
  "predictions": "/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/smoke_predictions_val_3.jsonl",
  "metrics": "/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/smoke_metrics_val_3.json"
}


/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py:49: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)
Loading weights: 100%|██████████| 947/947 [00:26<00:00, 35.25it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

echo "=== Smoke metrics ==="
python -m json.tool "$DRIVE_ROOT/baseline/run0_week5_checkpoint/smoke_metrics_val_3.json"

echo

echo "=== First prediction ==="
head -n 1 "$DRIVE_ROOT/baseline/run0_week5_checkpoint/smoke_predictions_val_3.jsonl" | python -m json.tool

=== Smoke metrics ===
{
    "created_at": "2026-06-16T05:01:47.948694+00:00",
    "phase": "phase3_run0_baseline",
    "eval_name": "smoke_val_3",
    "model": "/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint",
    "model_label": "week5_vietmed_checkpoint",
    "archive": "/content/drive/MyDrive/clinical_asr_vimedcss/subset_archives/run0_eval_val_200.tar.gz",
    "manifest": "/content/vimedcss_work/smoke_baseline_val/extracted/run0_eval_val_200/manifest.jsonl",
    "n_manifest_rows": 3,
    "n_prepared": 3,
    "n_skipped": 0,
    "skipped_preview": [],
    "strict_wer": 0.2518518518518518,
    "strict_cer": 0.18363939899833054,
    "normalized_wer": 0.17647058823529413,
    "normalized_cer": 0.16129032258064516,
    "total_runtime_seconds": 16.6336,
    "runtime_seconds_per_sample": 5.5445,
    "total_audio_hours": 0.0092,
    "language": "vi",
    "task": "transcribe",
    "effective_device": 0,
    "max_samples": 3,
    "does_not_train": true
}

=== Fi

### Phase 3G — Chạy validation 200

Đây là baseline chính để so sánh Run A/B/C.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python -u scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_val_200.tar.gz" \
  --work_dir "$WORK_ROOT/baseline_val" \
  --model "$MODEL" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_val_200.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_week5_checkpoint/metrics_val_200.json" \
  --clean_work_dir \
  --eval_name "run0_eval_val_200" \
  --model_label "week5_vietmed_checkpoint"

[INFO] Extracting: /content/drive/MyDrive/clinical_asr_vimedcss/subset_archives/run0_eval_val_200.tar.gz
[INFO] Manifest: /content/vimedcss_work/baseline_val/extracted/run0_eval_val_200/manifest.jsonl
[INFO] Loading ASR pipeline: /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
[INFO] processed 10/200
[INFO] processed 20/200
[INFO] processed 30/200
[INFO] processed 40/200
[INFO] processed 50/200
[INFO] processed 60/200
[INFO] processed 70/200
[INFO] processed 80/200
[INFO] processed 90/200
[INFO] processed 100/200
[INFO] processed 110/200
[INFO] processed 120/200
[INFO] processed 130/200
[INFO] processed 140/200
[INFO] processed 150/200
[INFO] processed 160/200
[INFO] processed 170/200
[INFO] processed 180/200
[INFO] processed 190/200
[INFO] processed 200/200
{
  "status": "DONE",
  "eval_name": "run0_eval_val_200",
  "n_prepared": 200,
  "n_skipped": 0,
  "strict_wer": 0.3247796278158668,
  "strict_cer": 0.22255141912490875,
  "normalized_wer": 0.25493646

/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py:49: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)
Loading weights: 100%|██████████| 947/947 [00:00<00:00, 1039.05it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcess

In [ ]:
%%writefile /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/asr_benchmark.py
#!/usr/bin/env python3
"""ASR benchmark"""

import argparse
import sys
import re
from collections import Counter
from dataclasses import dataclass

NEGATION_WORDS: frozenset[str] = frozenset({"không", "chưa", "chẳng", "chả", "đừng", "chớ"})


def levenshtein(ref: list, hyp: list) -> tuple[int, int, int]:
    """Return (substitutions, deletions, insertions) via dynamic programming."""
    n, m = len(ref), len(hyp)
    # dp[i][j] = (cost, s, d, i_count) for aligning ref[:i] with hyp[:j]
    dp = [[(0, 0, 0, 0)] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = (i, 0, i, 0)
    for j in range(1, m + 1):
        dp[0][j] = (j, 0, 0, j)

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                cost, s, d, ins = dp[i - 1][j - 1]
                dp[i][j] = (cost, s, d, ins)
            else:
                sub_cost, sub_s, sub_d, sub_i = dp[i - 1][j - 1]
                del_cost, del_s, del_d, del_i = dp[i - 1][j]
                ins_cost, ins_s, ins_d, ins_i = dp[i][j - 1]

                best = min(
                    (sub_cost + 1, sub_s + 1, sub_d, sub_i),
                    (del_cost + 1, del_s, del_d + 1, del_i),
                    (ins_cost + 1, ins_s, ins_d, ins_i + 1),
                )
                dp[i][j] = best

    _, s, d, ins = dp[n][m]
    return s, d, ins


@dataclass
class Metrics:
    wer: float
    cer: float
    substitutions: int
    deletions: int
    insertions: int
    ref_words: int
    ref_chars: int
    matched: int
    total_ref: int
    total_hyp: int
    cs_wer: float | None
    cs_substitutions: int
    cs_deletions: int
    cs_insertions: int
    cs_ref_words: int
    n_wer: float | None
    n_substitutions: int
    n_deletions: int
    n_insertions: int
    n_ref_words: int
    negation_miss_rate: float | None
    negation_missed: int
    negation_ref_count: int
    negation_utterances: int


def is_non_vietnamese(word: str, viet_set: frozenset[str]) -> bool:
    return word not in viet_set


def load_viet_syllables(path: str) -> frozenset[str]:
    with open(path, encoding="utf-8") as f:
        return frozenset(line.strip().lower() for line in f if line.strip())


def compute_metrics(
    references: dict[str, str],
    predictions: dict[str, str],
    viet_set: frozenset[str] | None = None,
) -> Metrics:
    total_s = total_d = total_i = 0
    total_ref_words = 0
    char_s = char_d = char_i = 0
    total_ref_chars = 0
    total_cs_s = total_cs_d = total_cs_i = 0
    total_cs_ref_words = 0
    total_n_s = total_n_d = total_n_i = 0
    total_n_ref_words = 0
    total_neg_missed = total_neg_ref = total_neg_utterances = 0
    matched = 0

    common_keys = set(references) & set(predictions)

    for key in common_keys:
        ref_words = references[key].split()
        hyp_words = predictions[key].split()
        s, d, ins = levenshtein(ref_words, hyp_words)
        total_s += s
        total_d += d
        total_i += ins
        total_ref_words += len(ref_words)

        ref_chars = list(references[key].replace(" ", ""))
        hyp_chars = list(predictions[key].replace(" ", ""))
        cs, cd, ci = levenshtein(ref_chars, hyp_chars)
        char_s += cs
        char_d += cd
        char_i += ci
        total_ref_chars += len(ref_chars)

        if viet_set is not None:
            ref_cs = [w for w in ref_words if is_non_vietnamese(w, viet_set)]
            hyp_cs = [w for w in hyp_words if is_non_vietnamese(w, viet_set)]
            cs_s, cs_d, cs_i = levenshtein(ref_cs, hyp_cs)
            total_cs_s += cs_s
            total_cs_d += cs_d
            total_cs_i += cs_i
            total_cs_ref_words += len(ref_cs)

            ref_n = [w for w in ref_words if not is_non_vietnamese(w, viet_set)]
            hyp_n = [w for w in hyp_words if not is_non_vietnamese(w, viet_set)]
            n_s, n_d, n_i = levenshtein(ref_n, hyp_n)
            total_n_s += n_s
            total_n_d += n_d
            total_n_i += n_i
            total_n_ref_words += len(ref_n)

        ref_neg = Counter(w for w in ref_words if w in NEGATION_WORDS)
        hyp_neg = Counter(w for w in hyp_words if w in NEGATION_WORDS)
        utterance_missed = sum(max(0, ref_neg[w] - hyp_neg[w]) for w in ref_neg)
        total_neg_missed += utterance_missed
        total_neg_ref += sum(ref_neg.values())
        if ref_neg:
            total_neg_utterances += 1

        matched += 1
    wer =(total_s + total_d + total_i) / total_ref_words if total_ref_words > 0 else 0.0
    cer = (char_s + char_d + char_i) / total_ref_chars if total_ref_chars > 0 else 0.0
    cs_wer = (
        (total_cs_s + total_cs_d + total_cs_i) / total_cs_ref_words
        if viet_set is not None and total_cs_ref_words > 0
        else (0.0 if viet_set is not None else None)
    )
    n_wer = (
        (total_n_s + total_n_d + total_n_i) / total_n_ref_words
        if viet_set is not None and total_n_ref_words > 0
        else (0.0 if viet_set is not None else None)
    )
    negation_miss_rate = total_neg_missed / total_neg_ref if total_neg_ref > 0 else None

    return Metrics(
        wer=wer,
        cer=cer,
        substitutions=total_s,
        deletions=total_d,
        insertions=total_i,
        ref_words=total_ref_words,
        ref_chars=total_ref_chars,
        matched=matched,
        total_ref=len(references),
        total_hyp=len(predictions),
        cs_wer=cs_wer,
        cs_substitutions=total_cs_s,
        cs_deletions=total_cs_d,
        cs_insertions=total_cs_i,
        cs_ref_words=total_cs_ref_words,
        n_wer=n_wer,
        n_substitutions=total_n_s,
        n_deletions=total_n_d,
        n_insertions=total_n_i,
        n_ref_words=total_n_ref_words,
        negation_miss_rate=negation_miss_rate,
        negation_missed=total_neg_missed,
        negation_ref_count=total_neg_ref,
        negation_utterances=total_neg_utterances,
    )

tone_mapper = {
    'òa': 'oà',
    'óa': 'oá',
    'ỏa': 'oả',
    'õa': 'oã',
    'ọa': 'oạ',
    'òe': 'oè',
    'óe': 'oé',
    'ỏe': 'oẻ',
    'õe': 'oẽ',
    'ọe': 'oẹ',
    'ùy': 'uỳ',
    'úy': 'uý',
    'ủy': 'uỷ',
    'ũy': 'uỹ',
    'ụy': 'uỵ',
}


def normalize_text(text):
    # Remove language tags like <vi-VN>, <en-US>, etc.
    text = re.sub(r'<[a-zA-Z]{2}-[a-zA-Z]{2,}>', '', text)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    # normalize punctuator
    text = re.sub(r'\s[\.|\,|\:|\(|\)|\?|\!|\~|\%]+', ' ', text)
    text = re.sub(r'[\.|\,|\:|\(|\)|\?|\!|\~|\%]+\s', ' ', text)

    # normalize tone
    for key in tone_mapper.keys():
        text = text.replace(key, tone_mapper[key])

    text = text.strip()
    return text


def load_file(path: str) -> dict[str, str]:
    """Load tab-separated file: filename<TAB>text, keyed by filename."""
    data = {}
    with open(path, encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                print(f"Warning: {path}:{lineno} has fewer than 2 columns, skipping", file=sys.stderr)
                continue
            filename = parts[0].strip()
            text = normalize_text(parts[1].strip().lower())
            if filename in data:
                print(f"Warning: duplicate filename '{filename}' in {path}:{lineno}", file=sys.stderr)
            data[filename] = text
    return data


def main():
    parser = argparse.ArgumentParser(description="Benchmark ASR model: compute WER and CER")
    parser.add_argument("--prediction", help="Path to prediction file (filename<TAB>text)")
    parser.add_argument("--transcription", help="Path to reference transcription file (filename<TAB>text)")
    parser.add_argument(
        "--viet-syllables",
        metavar="FILE",
        help="Path to Vietnamese syllable list (one per line). "
             "Download from https://gist.github.com/hieuthi/0f5adb7d3f79e7fb67e0e499004bf558",
    )
    args = parser.parse_args()

    references = load_file(args.transcription)
    predictions = load_file(args.prediction)

    viet_set = load_viet_syllables(args.viet_syllables) if args.viet_syllables else None
    metrics = compute_metrics(references, predictions, viet_set)

    missing_in_pred = set(references) - set(predictions)
    missing_in_ref = set(predictions) - set(references)

    print(f"{'='*50}")
    print(f"  ASR Benchmark Results")
    print(f"{'='*50}")
    print(f"  Files matched:        {metrics.matched}")
    print(f"  Reference total:      {metrics.total_ref}")
    print(f"  Prediction total:     {metrics.total_hyp}")
    if missing_in_pred:
        print(f"  Missing in pred:      {len(missing_in_pred)}")
    if missing_in_ref:
        print(f"  Missing in ref:       {len(missing_in_ref)}")
    print(f"{'-'*50}")
    print(f"  WER:                  {metrics.wer * 100:.2f}%")
    print(f"  CER:                  {metrics.cer * 100:.2f}%")
    if metrics.cs_wer is not None:
        print(f"  CS-WER:               {metrics.cs_wer * 100:.2f}%")
        print(f"  N-WER:                {metrics.n_wer * 100:.2f}%")
    else:
        print(f"  CS-WER:               N/A  (use --viet-syllables to enable)")
        print(f"  N-WER:                N/A  (use --viet-syllables to enable)")
    if metrics.negation_miss_rate is not None:
        print(
            f"  Negation miss rate:   {metrics.negation_miss_rate * 100:.2f}%"
            f"  ({metrics.negation_missed} missed / {metrics.negation_ref_count} ref negations,"
            f" {metrics.negation_utterances} utterances)"
        )
    else:
        print(f"  Negation miss rate:   N/A  (no negation words in reference)")
    print(f"{'-'*50}")
    print(f"  Word errors:          {metrics.substitutions + metrics.deletions + metrics.insertions}")
    print(f"    Substitutions:      {metrics.substitutions}")
    print(f"    Deletions:          {metrics.deletions}")
    print(f"    Insertions:         {metrics.insertions}")
    print(f"  Reference words:      {metrics.ref_words}")
    print(f"  Reference chars:      {metrics.ref_chars}")
    if metrics.cs_wer is not None:
        print(f"{'-'*50}")
        print(f"  CS word errors:       {metrics.cs_substitutions + metrics.cs_deletions + metrics.cs_insertions}")
        print(f"    Substitutions:      {metrics.cs_substitutions}")
        print(f"    Deletions:          {metrics.cs_deletions}")
        print(f"    Insertions:         {metrics.cs_insertions}")
        print(f"  CS ref words:         {metrics.cs_ref_words}")
        print(f"  N word errors:        {metrics.n_substitutions + metrics.n_deletions + metrics.n_insertions}")
        print(f"    Substitutions:      {metrics.n_substitutions}")
        print(f"    Deletions:          {metrics.n_deletions}")
        print(f"    Insertions:         {metrics.n_insertions}")
        print(f"  N ref words:          {metrics.n_ref_words}")
    print(f"{'='*50}")


if __name__ == "__main__":
    main()

Writing /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/asr_benchmark.py


In [ ]:
%%bash
set -e

chmod +x /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/asr_benchmark.py

##### Convert prediction JSONL của phase 3G sang TSV

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

export PRED_JSONL="$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_val_200.jsonl"
export BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
export REF_TSV="$BENCH_DIR/val_200_reference.tsv"
export HYP_TSV="$BENCH_DIR/val_200_prediction.tsv"

mkdir -p "$BENCH_DIR"

python - <<'PY'
import json
import os
from pathlib import Path

pred_jsonl = Path(os.environ["PRED_JSONL"])
ref_tsv = Path(os.environ["REF_TSV"])
hyp_tsv = Path(os.environ["HYP_TSV"])

if not pred_jsonl.exists():
    raise FileNotFoundError(f"Missing predictions JSONL: {pred_jsonl}")

n = 0
with pred_jsonl.open("r", encoding="utf-8") as f, \
     ref_tsv.open("w", encoding="utf-8") as rf, \
     hyp_tsv.open("w", encoding="utf-8") as hf:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)

        sid = (
            row.get("segment_id")
            or row.get("sample_id")
            or row.get("id")
            or f"sample_{n:06d}"
        )

        ref = (
            row.get("reference_text")
            or row.get("segment_text")
            or row.get("sentence")
            or ""
        )

        hyp = (
            row.get("prediction_text")
            or row.get("predicted_text")
            or row.get("prediction")
            or row.get("text")
            or ""
        )

        sid = str(sid).replace("\t", " ").strip()
        ref = str(ref).replace("\t", " ").replace("\n", " ").strip()
        hyp = str(hyp).replace("\t", " ").replace("\n", " ").strip()

        rf.write(f"{sid}\t{ref}\n")
        hf.write(f"{sid}\t{hyp}\n")
        n += 1

print(f"[DONE] Converted {n} rows")
print(f"REF_TSV={ref_tsv}")
print(f"HYP_TSV={hyp_tsv}")
PY

[DONE] Converted 200 rows
REF_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/val_200_reference.tsv
HYP_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/val_200_prediction.tsv


##### Chạy benchmark với validation 200

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
REF_TSV="$BENCH_DIR/val_200_reference.tsv"
HYP_TSV="$BENCH_DIR/val_200_prediction.tsv"
OUT_TXT="$BENCH_DIR/asr_benchmark_val_200.txt"

cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/asr_benchmark.py \
  --prediction "$HYP_TSV" \
  --transcription "$REF_TSV" \
  --viet-syllables "scripts/asr_vimedcss/viet_syllables.txt" | tee "$OUT_TXT"

echo
echo "[DONE] Benchmark saved to:"
echo "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  26.09%
  CER:                  21.64%
  CS-WER:               26.09%
  N-WER:                0.00%
  Negation miss rate:   12.82%  (5 missed / 39 ref negations, 35 utterances)
--------------------------------------------------
  Word errors:          1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  Reference words:      5105
  Reference chars:      18169
--------------------------------------------------
  CS word errors:       1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  CS ref words:         5105
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0

[DONE] Benchmark saved to:
/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_ch

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"

mkdir -p "$SCRIPT_DIR"

VIET_SYLLABLES_URL="https://gist.githubusercontent.com/hieuthi/0f5adb7d3f79e7fb67e0e499004bf558/raw/e304918e590059c3a3c2602737f14b3cfc23318f/viet_syllables.txt"
VIET_SYLLABLES_PATH="$SCRIPT_DIR/viet_syllables.txt"

if [ ! -f "$VIET_SYLLABLES_PATH" ]; then
  echo "[INFO] Downloading Vietnamese syllables list..."
  wget -O "$VIET_SYLLABLES_PATH" "$VIET_SYLLABLES_URL"
  echo "[OK] Downloaded $VIET_SYLLABLES_PATH"
else
  echo "[INFO] Vietnamese syllables list already exists: $VIET_SYLLABLES_PATH"
fi

ls -l "$VIET_SYLLABLES_PATH"

[INFO] Vietnamese syllables list already exists: /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/viet_syllables.txt
-rw-r--r-- 1 root root 0 Jun 16 05:18 /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/viet_syllables.txt


In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
REF_TSV="$BENCH_DIR/val_200_reference.tsv"
HYP_TSV="$BENCH_DIR/val_200_prediction.tsv"
OUT_TXT="$BENCH_DIR/asr_benchmark_val_200.txt"

cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/asr_benchmark.py \
  --prediction "$HYP_TSV" \
  --transcription "$REF_TSV" \
  --viet-syllables "scripts/asr_vimedcss/viet_syllables.txt" | tee "$OUT_TXT"

echo
echo "[DONE] Benchmark saved to:"
echo "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  26.09%
  CER:                  21.64%
  CS-WER:               26.09%
  N-WER:                0.00%
  Negation miss rate:   12.82%  (5 missed / 39 ref negations, 35 utterances)
--------------------------------------------------
  Word errors:          1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  Reference words:      5105
  Reference chars:      18169
--------------------------------------------------
  CS word errors:       1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  CS ref words:         5105
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0

[DONE] Benchmark saved to:
/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_ch

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
OUT_TXT="$BENCH_DIR/asr_benchmark_val_200.txt"

cat "$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark/asr_benchmark_val_200.txt"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  26.09%
  CER:                  21.64%
  CS-WER:               26.09%
  N-WER:                0.00%
  Negation miss rate:   12.82%  (5 missed / 39 ref negations, 35 utterances)
--------------------------------------------------
  Word errors:          1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  Reference words:      5105
  Reference chars:      18169
--------------------------------------------------
  CS word errors:       1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  CS ref words:         5105
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0


In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_hard_200.tar.gz" \
  --work_dir "$WORK_ROOT/baseline_hard" \
  --model "$MODEL" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_week5_checkpoint/metrics_hard_200.json" \
  --clean_work_dir \
  --eval_name "run0_eval_hard_200" \
  --model_label "week5_vietmed_checkpoint"

[INFO] Extracting: /content/drive/MyDrive/clinical_asr_vimedcss/subset_archives/run0_eval_hard_200.tar.gz
[INFO] Manifest: /content/vimedcss_work/baseline_hard/extracted/run0_eval_hard_200/manifest.jsonl
[INFO] Loading ASR pipeline: /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
[INFO] processed 10/200
[INFO] processed 20/200
[INFO] processed 30/200
[INFO] processed 40/200
[INFO] processed 50/200
[INFO] processed 60/200
[INFO] processed 70/200
[INFO] processed 80/200
[INFO] processed 90/200
[INFO] processed 100/200
[INFO] processed 110/200
[INFO] processed 120/200
[INFO] processed 130/200
[INFO] processed 140/200
[INFO] processed 150/200
[INFO] processed 160/200
[INFO] processed 170/200
[INFO] processed 180/200
[INFO] processed 190/200
[INFO] processed 200/200
{
  "status": "DONE",
  "eval_name": "run0_eval_hard_200",
  "n_prepared": 200,
  "n_skipped": 0,
  "strict_wer": 0.3859546165884194,
  "strict_cer": 0.25142857142857145,
  "normalized_wer": 0.3112

/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py:49: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)
Loading weights: 100%|██████████| 947/947 [00:00<00:00, 2479.36it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcess

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

export PRED_JSONL="$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl"
export BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
export REF_TSV="$BENCH_DIR/hard_200_reference.tsv"
export HYP_TSV="$BENCH_DIR/hard_200_prediction.tsv"

mkdir -p "$BENCH_DIR"

python - <<'PY'
import json
import os
from pathlib import Path

pred_jsonl = Path(os.environ["PRED_JSONL"])
ref_tsv = Path(os.environ["REF_TSV"])
hyp_tsv = Path(os.environ["HYP_TSV"])

if not pred_jsonl.exists():
    raise FileNotFoundError(f"Missing predictions JSONL: {pred_jsonl}")

n = 0
with pred_jsonl.open("r", encoding="utf-8") as f, \
     ref_tsv.open("w", encoding="utf-8") as rf, \
     hyp_tsv.open("w", encoding="utf-8") as hf:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)

        sid = (
            row.get("segment_id")
            or row.get("sample_id")
            or row.get("id")
            or f"sample_{n:06d}"
        )

        ref = (
            row.get("reference_text")
            or row.get("segment_text")
            or row.get("sentence")
            or ""
        )

        hyp = (
            row.get("prediction_text")
            or row.get("predicted_text")
            or row.get("prediction")
            or row.get("text")
            or ""
        )

        sid = str(sid).replace("\t", " ").strip()
        ref = str(ref).replace("\t", " ").replace("\n", " ").strip()
        hyp = str(hyp).replace("\t", " ").replace("\n", " ").strip()

        rf.write(f"{sid}\t{ref}\n")
        hf.write(f"{sid}\t{hyp}\n")
        n += 1

print(f"[DONE] Converted {n} rows")
print(f"REF_TSV={ref_tsv}")
print(f"HYP_TSV={hyp_tsv}")
PY

[DONE] Converted 200 rows
REF_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_reference.tsv
HYP_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_prediction.tsv


In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
REF_TSV="$BENCH_DIR/hard_200_reference.tsv"
HYP_TSV="$BENCH_DIR/hard_200_prediction.tsv"
OUT_TXT="$BENCH_DIR/asr_benchmark_hard_200.txt"

cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/asr_benchmark.py \
  --prediction "$HYP_TSV" \
  --transcription "$REF_TSV" \
  --viet-syllables "scripts/asr_vimedcss/viet_syllables.txt" | tee "$OUT_TXT"

echo
echo "[DONE] Benchmark saved to:"
echo "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  32.14%
  CER:                  24.29%
  CS-WER:               32.14%
  N-WER:                0.00%
  Negation miss rate:   11.36%  (5 missed / 44 ref negations, 30 utterances)
--------------------------------------------------
  Word errors:          1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  Reference words:      5112
  Reference chars:      18807
--------------------------------------------------
  CS word errors:       1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  CS ref words:         5112
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0

[DONE] Benchmark saved to:
/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_ch

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
OUT_TXT="$BENCH_DIR/asr_benchmark_hard_200.txt"

cat "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  32.14%
  CER:                  24.29%
  CS-WER:               32.14%
  N-WER:                0.00%
  Negation miss rate:   11.36%  (5 missed / 44 ref negations, 30 utterances)
--------------------------------------------------
  Word errors:          1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  Reference words:      5112
  Reference chars:      18807
--------------------------------------------------
  CS word errors:       1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  CS ref words:         5112
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0


In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
SCRIPT_DIR="$PROJECT_ROOT/scripts/asr_vimedcss"

mkdir -p "$SCRIPT_DIR"

VIET_SYLLABLES_PATH="$SCRIPT_DIR/viet_syllables.txt"
TMP_PATH="$VIET_SYLLABLES_PATH.tmp"

# Candidate URLs.
CANDIDATE_URLS=(
  "https://raw.githubusercontent.com/vietnameselanguage/syllable/master/vietnamesesyllable.txt"
  "https://raw.githubusercontent.com/vietnameselanguage/syllable/master/vietnamesesyllable_7184.txt"
  "https://gist.githubusercontent.com/hieuthi/1f5d80fca871f3642f61f7e3de883f3a/raw/common-vietnamese-syllables.txt"
)

# Remove existing file to ensure a clean state.
rm -f "$VIET_SYLLABLES_PATH"

echo "[INFO] Downloading Vietnamese syllables list..."
SUCCESS=0

for URL in "${CANDIDATE_URLS[@]}"; do
  echo "[INFO] Trying: $URL"
  rm -f "$TMP_PATH"

  if curl -fL --retry 3 --connect-timeout 20 "$URL" -o "$TMP_PATH"; then
    if [ -s "$TMP_PATH" ]; then
      # Fix encoding: convert to UTF-8 if it is UTF-16 or has a BOM
      # We use 'iconv' to ensure the output is pure UTF-8
      if file "$TMP_PATH" | grep -q "UTF-16"; then
         echo "[INFO] Converting from UTF-16 to UTF-8..."
         iconv -f UTF-16 -t UTF-8 "$TMP_PATH" > "$VIET_SYLLABLES_PATH"
      else
         cat "$TMP_PATH" > "$VIET_SYLLABLES_PATH"
      fi

      LINE_COUNT=$(wc -l < "$VIET_SYLLABLES_PATH" | tr -d ' ')
      if [ "$LINE_COUNT" -gt 1000 ]; then
        echo "[OK] Downloaded and processed syllable list from: $URL"
        echo "[OK] Line count: $LINE_COUNT"
        SUCCESS=1
        break
      fi
    fi
  fi
done

rm -f "$TMP_PATH"

if [ "$SUCCESS" -ne 1 ]; then
  echo "[ERROR] Could not download a valid Vietnamese syllable list."
  exit 1
fi

echo "[INFO] Final file info:"
ls -lh "$VIET_SYLLABLES_PATH"
head -n 5 "$VIET_SYLLABLES_PATH"

[INFO] Downloading Vietnamese syllables list...
[INFO] Trying: https://raw.githubusercontent.com/vietnameselanguage/syllable/master/vietnamesesyllable.txt
[INFO] Converting from UTF-16 to UTF-8...
[OK] Downloaded and processed syllable list from: https://raw.githubusercontent.com/vietnameselanguage/syllable/master/vietnamesesyllable.txt
[OK] Line count: 6674
[INFO] Final file info:
-rw-r--r-- 1 root root 46K Jun 16 05:45 /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/viet_syllables.txt
a
à
ả
ã
á


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 73210  100 73210    0     0   343k      0 --:--:-- --:--:-- --:--:--  343k


### Kết quả benchmark cho validation 200

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
OUT_TXT="$BENCH_DIR/asr_benchmark_val_200.txt"

cat "$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark/asr_benchmark_val_200.txt"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  26.09%
  CER:                  21.64%
  CS-WER:               26.09%
  N-WER:                0.00%
  Negation miss rate:   12.82%  (5 missed / 39 ref negations, 35 utterances)
--------------------------------------------------
  Word errors:          1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  Reference words:      5105
  Reference chars:      18169
--------------------------------------------------
  CS word errors:       1332
    Substitutions:      520
    Deletions:          471
    Insertions:         341
  CS ref words:         5105
  N word errors:        0
    Substitutions:      0
    Deletions:          0
    Insertions:         0
  N ref words:          0


Tiếp theo, bạn có thể chạy benchmark cho tập dữ liệu `hard_200`.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

export PRED_JSONL="$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl"
export BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
export REF_TSV="$BENCH_DIR/hard_200_reference.tsv"
export HYP_TSV="$BENCH_DIR/hard_200_prediction.tsv"

mkdir -p "$BENCH_DIR"

python - <<'PY'
import json
import os
from pathlib import Path

pred_jsonl = Path(os.environ["PRED_JSONL"])
ref_tsv = Path(os.environ["REF_TSV"])
hyp_tsv = Path(os.environ["HYP_TSV"])

if not pred_jsonl.exists():
    raise FileNotFoundError(f"Missing predictions JSONL: {pred_jsonl}")

n = 0
with pred_jsonl.open("r", encoding="utf-8") as f, \
     ref_tsv.open("w", encoding="utf-8") as rf, \
     hyp_tsv.open("w", encoding="utf-8") as hf:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)

        sid = (
            row.get("segment_id")
            or row.get("sample_id")
            or row.get("id")
            or f"sample_{n:06d}"
        )

        ref = (
            row.get("reference_text")
            or row.get("segment_text")
            or row.get("sentence")
            or ""
        )

        hyp = (
            row.get("prediction_text")
            or row.get("predicted_text")
            or row.get("prediction")
            or row.get("text")
            or ""
        )

        sid = str(sid).replace("\t", " ").strip()
        ref = str(ref).replace("\t", " ").replace("\n", " ").strip()
        hyp = str(hyp).replace("\t", " ").replace("\n", " ").strip()

        rf.write(f"{sid}\t{ref}\n")
        hf.write(f"{sid}\t{hyp}\n")
        n += 1

print(f"[DONE] Converted {n} rows")
print(f"REF_TSV={ref_tsv}")
print(f"HYP_TSV={hyp_tsv}")
PY

[DONE] Converted 200 rows
REF_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_reference.tsv
HYP_TSV=/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_prediction.tsv


Chạy benchmark cho `hard_200`:

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
REF_TSV="$BENCH_DIR/hard_200_reference.tsv"
HYP_TSV="$BENCH_DIR/hard_200_prediction.tsv"
OUT_TXT="$BENCH_DIR/asr_benchmark_hard_200.txt"

cd "$PROJECT_ROOT"

# Re-running benchmark with the now-fixed UTF-8 syllable list
python scripts/asr_vimedcss/asr_benchmark.py \
  --prediction "$HYP_TSV" \
  --transcription "$REF_TSV" \
  --viet-syllables "scripts/asr_vimedcss/viet_syllables.txt" | tee "$OUT_TXT"

echo
echo "[DONE] Benchmark saved to:"
echo "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  32.14%
  CER:                  24.29%
  CS-WER:               96.14%
  N-WER:                28.04%
  Negation miss rate:   11.36%  (5 missed / 44 ref negations, 30 utterances)
--------------------------------------------------
  Word errors:          1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  Reference words:      5112
  Reference chars:      18807
--------------------------------------------------
  CS word errors:       673
    Substitutions:      190
    Deletions:          414
    Insertions:         69
  CS ref words:         700
  N word errors:        1237
    Substitutions:      269
    Deletions:          270
    Insertions:         698
  N ref words:          4412

[DONE] Benchmark saved to:
/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run

Sau khi chạy xong, bạn có thể xem kết quả benchmark cho `hard_200` bằng cách chạy cell sau:

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
OUT_TXT="$BENCH_DIR/asr_benchmark_hard_200.txt"

cat "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  32.14%
  CER:                  24.29%
  CS-WER:               N/A  (use --viet-syllables to enable)
  N-WER:                N/A  (use --viet-syllables to enable)
  Negation miss rate:   11.36%  (5 missed / 44 ref negations, 30 utterances)
--------------------------------------------------
  Word errors:          1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  Reference words:      5112
  Reference chars:      18807


In [ ]:
import os
from pathlib import Path

syllable_path = Path('/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/viet_syllables.txt')

if syllable_path.exists():
    size = syllable_path.stat().st_size
    print(f'File size: {size} bytes')
    with open(syllable_path, 'r', encoding='utf-8') as f:
        lines = [next(f).strip() for _ in range(5)]
    print(f'First 5 syllables: {lines}')
else:
    print('ERROR: Syllable file missing!')

File size: 46878 bytes
First 5 syllables: ['a', 'à', 'ả', 'ã', 'á']


In [ ]:
%%bash
set -euo pipefail
source /content/phase3_env.sh

# Define paths
BENCH_DIR="$DRIVE_ROOT/baseline/run0_week5_checkpoint/mentor_asr_benchmark"
REF_TSV="$BENCH_DIR/hard_200_reference.tsv"
HYP_TSV="$BENCH_DIR/hard_200_prediction.tsv"
OUT_TXT="$BENCH_DIR/asr_benchmark_hard_200.txt"
SYLLABLES="/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/viet_syllables.txt"

# Run the benchmark script and force overwrite the output file
python /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/asr_benchmark.py \
  --prediction "$HYP_TSV" \
  --transcription "$REF_TSV" \
  --viet-syllables "$SYLLABLES" > "$OUT_TXT"

# Display the fresh result
cat "$OUT_TXT"

  ASR Benchmark Results
  Files matched:        200
  Reference total:      200
  Prediction total:     200
--------------------------------------------------
  WER:                  32.14%
  CER:                  24.29%
  CS-WER:               96.14%
  N-WER:                28.04%
  Negation miss rate:   11.36%  (5 missed / 44 ref negations, 30 utterances)
--------------------------------------------------
  Word errors:          1643
    Substitutions:      726
    Deletions:          417
    Insertions:         500
  Reference words:      5112
  Reference chars:      18807
--------------------------------------------------
  CS word errors:       673
    Substitutions:      190
    Deletions:          414
    Insertions:         69
  CS ref words:         700
  N word errors:        1237
    Substitutions:      269
    Deletions:          270
    Insertions:         698
  N ref words:          4412


### Phase 3H — Chạy hard 200

Hard 200 chỉ dùng để reporting/stress, không dùng để chọn checkpoint hoặc hyperparameter.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_hard_200.tar.gz" \
  --work_dir "$WORK_ROOT/baseline_hard" \
  --model "$MODEL" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_week5_checkpoint/metrics_hard_200.json" \
  --clean_work_dir \
  --eval_name "run0_eval_hard_200" \
  --model_label "week5_vietmed_checkpoint"

[INFO] Extracting: /content/drive/MyDrive/clinical_asr_vimedcss/subset_archives/run0_eval_hard_200.tar.gz
[INFO] Manifest: /content/vimedcss_work/baseline_hard/extracted/run0_eval_hard_200/manifest.jsonl
[INFO] Loading ASR pipeline: /content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
[INFO] processed 10/200
[INFO] processed 20/200
[INFO] processed 30/200
[INFO] processed 40/200
[INFO] processed 50/200
[INFO] processed 60/200
[INFO] processed 70/200
[INFO] processed 80/200
[INFO] processed 90/200
[INFO] processed 100/200
[INFO] processed 110/200
[INFO] processed 120/200
[INFO] processed 130/200
[INFO] processed 140/200
[INFO] processed 150/200
[INFO] processed 160/200
[INFO] processed 170/200
[INFO] processed 180/200
[INFO] processed 190/200
[INFO] processed 200/200
{
  "status": "DONE",
  "eval_name": "run0_eval_hard_200",
  "n_prepared": 200,
  "n_skipped": 0,
  "strict_wer": 0.3859546165884194,
  "strict_cer": 0.25142857142857145,
  "normalized_wer": 0.3112

/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py:49: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)
Loading weights: 100%|██████████| 947/947 [00:00<00:00, 2052.26it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcess

### Phase 3I — Optional: chạy test 200 reporting-only

Mặc định cell dưới **không chạy test**. Nếu muốn chạy theo plan, đổi:

```bash
RUN_TEST="1"
```

Rule bắt buộc:

```text
Không dùng test để chọn run.
Không dùng test để chỉnh hyperparameter.
Không dùng test để quyết định Run A/B/C.
Chỉ reporting-only.
```

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

RUN_TEST="0"

if [ "$RUN_TEST" != "1" ]; then
  echo "[SKIP] Test 200 reporting-only is intentionally skipped. Set RUN_TEST=1 to run."
  exit 0
fi

python scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_test_200.tar.gz" \
  --work_dir "$WORK_ROOT/baseline_test" \
  --model "$MODEL" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_test_200.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_week5_checkpoint/metrics_test_200.json" \
  --clean_work_dir \
  --eval_name "run0_eval_test_200_reporting_only" \
  --model_label "week5_vietmed_checkpoint"

[SKIP] Test 200 reporting-only is intentionally skipped. Set RUN_TEST=1 to run.


### Phase 3J — Optional: original PhoWhisper validation baseline

Mặc định không chạy để tiết kiệm GPU. Nếu còn GPU time và muốn so sánh checkpoint Week 5 với model gốc, đổi `RUN_ORIGINAL="1"`.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

RUN_ORIGINAL="0"

if [ "$RUN_ORIGINAL" != "1" ]; then
  echo "[SKIP] Original PhoWhisper baseline is intentionally skipped. Set RUN_ORIGINAL=1 to run."
  exit 0
fi

python scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py \
  --archive "$DRIVE_ROOT/subset_archives/run0_eval_val_200.tar.gz" \
  --work_dir "$WORK_ROOT/baseline_original_val" \
  --model "vinai/PhoWhisper-medium" \
  --output_predictions "$DRIVE_ROOT/baseline/run0_original_phowhisper/predictions_val_200.jsonl" \
  --output_metrics "$DRIVE_ROOT/baseline/run0_original_phowhisper/metrics_val_200.json" \
  --clean_work_dir \
  --eval_name "run0_original_phowhisper_val_200" \
  --model_label "original_phowhisper_medium"

[SKIP] Original PhoWhisper baseline is intentionally skipped. Set RUN_ORIGINAL=1 to run.


### Phase 3K — Tạo report script Run 0

Report tổng hợp metrics của Week 5 checkpoint trên validation/hard/test và optional original PhoWhisper.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

SCRIPT_DIR="scripts/asr_vimedcss"
mkdir -p "$SCRIPT_DIR"
SCRIPT="$SCRIPT_DIR/03_make_run0_baseline_report.py"

cat > "$SCRIPT" <<'PYCODE'
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
from __future__ import annotations
import argparse, json
from pathlib import Path
from typing import Any

def read_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def pct(v: Any) -> str:
    if v is None:
        return "TBD"
    try:
        return f"{float(v) * 100:.2f}%"
    except Exception:
        return "TBD"

def row(model: str, eval_set: str, path: Path, notes: str) -> dict[str, Any]:
    m = read_json(path)
    if m is None:
        return {"model": model, "eval_set": eval_set, "samples": "missing", "wer": "TBD", "cer": "TBD", "nwer": "TBD", "ncer": "TBD", "runtime": "TBD", "notes": notes, "status": "missing", "path": str(path)}
    return {"model": model, "eval_set": eval_set, "samples": m.get("n_prepared"), "wer": pct(m.get("strict_wer")), "cer": pct(m.get("strict_cer")), "nwer": pct(m.get("normalized_wer")), "ncer": pct(m.get("normalized_cer")), "runtime": m.get("runtime_seconds_per_sample"), "notes": notes, "status": "present", "path": str(path)}

def render(rows: list[dict[str, Any]], drive_root: Path) -> str:
    lines = []
    lines.append("# Phase 3 — Run 0 ViMedCSS Baseline Report\n")
    lines.append("## Objective\n")
    lines.append("Measure the Week 5 VietMed PhoWhisper checkpoint on ViMedCSS subsets before ViMedCSS fine-tuning. This is baseline only; no training is performed.\n")
    lines.append("## Results\n")
    lines.append("| Model | Eval set | Samples | Strict WER | Strict CER | Normalized WER | Normalized CER | Runtime/sample | Notes |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---|")
    for r in rows:
        lines.append(f"| {r['model']} | {r['eval_set']} | {r['samples']} | {r['wer']} | {r['cer']} | {r['nwer']} | {r['ncer']} | {r['runtime']} | {r['notes']} |")
    lines.append("\n## Metric files\n")
    lines.append("| Model | Eval set | Status | Metrics path |")
    lines.append("|---|---|---|---|")
    for r in rows:
        lines.append(f"| {r['model']} | {r['eval_set']} | {r['status']} | `{r['path']}` |")
    lines.append("\n## Interpretation\n")
    lines.append("Use validation 200 as the main Run 0 baseline for comparing Run A/B/C. Hard 200 is reporting/stress only. Test 200, if executed, is reporting-only and must not be used to choose hyperparameters or checkpoints.\n")
    lines.append("## Gate to Phase 4\n")
    lines.append("Proceed to Run A only if validation baseline completed, predictions/metrics exist, audio path resolution worked, and predictions are not systematically empty.\n")
    lines.append("## Drive root\n")
    lines.append(f"`{drive_root}`\n")
    return "\n".join(lines)

def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--drive_root", required=True)
    ap.add_argument("--output_report", required=True)
    args = ap.parse_args()
    drive = Path(args.drive_root)
    rows = [
        row("Week5 checkpoint", "validation 200", drive / "baseline/run0_week5_checkpoint/metrics_val_200.json", "model-selection baseline"),
        row("Week5 checkpoint", "hard 200", drive / "baseline/run0_week5_checkpoint/metrics_hard_200.json", "reporting/stress only"),
        row("Week5 checkpoint", "test 200", drive / "baseline/run0_week5_checkpoint/metrics_test_200.json", "reporting-only if executed"),
        row("Original PhoWhisper", "validation 200", drive / "baseline/run0_original_phowhisper/metrics_val_200.json", "optional comparison"),
    ]
    out = Path(args.output_report)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(render(rows, drive), encoding="utf-8")
    print(json.dumps({"status": "DONE", "output_report": str(out), "rows": rows}, ensure_ascii=False, indent=2))

if __name__ == "__main__":
    main()
PYCODE

chmod +x "$SCRIPT"
echo "[OK] Created $SCRIPT"
python "$SCRIPT" --help

[OK] Created scripts/asr_vimedcss/03_make_run0_baseline_report.py
usage: 03_make_run0_baseline_report.py [-h] --drive_root DRIVE_ROOT
                                       --output_report OUTPUT_REPORT

options:
  -h, --help            show this help message and exit
  --drive_root DRIVE_ROOT
  --output_report OUTPUT_REPORT


### Phase 3L — Tạo report tổng hợp Run 0

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh
cd "$PROJECT_ROOT"

python scripts/asr_vimedcss/03_make_run0_baseline_report.py \
  --drive_root "$DRIVE_ROOT" \
  --output_report "$DRIVE_ROOT/baseline/run0_week5_checkpoint/BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md"

mkdir -p "$PROJECT_ROOT/experiments/asr/vimedcss/baseline/run0_week5_checkpoint"
cp "$DRIVE_ROOT/baseline/run0_week5_checkpoint/BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md" \
   "$PROJECT_ROOT/experiments/asr/vimedcss/baseline/run0_week5_checkpoint/"

cat "$DRIVE_ROOT/baseline/run0_week5_checkpoint/BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md"

{
  "status": "DONE",
  "output_report": "/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md",
  "rows": [
    {
      "model": "Week5 checkpoint",
      "eval_set": "validation 200",
      "samples": 200,
      "wer": "32.48%",
      "cer": "22.26%",
      "nwer": "25.49%",
      "ncer": "19.98%",
      "runtime": 3.2256,
      "notes": "model-selection baseline",
      "status": "present",
      "path": "/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/metrics_val_200.json"
    },
    {
      "model": "Week5 checkpoint",
      "eval_set": "hard 200",
      "samples": 200,
      "wer": "38.60%",
      "cer": "25.14%",
      "nwer": "31.13%",
      "ncer": "22.55%",
      "runtime": 3.3751,
      "notes": "reporting/stress only",
      "status": "present",
      "path": "/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/metrics_hard_200.json"
    },
    {
      "model"

### Phase 3M — Kiểm tra output cuối Phase 3

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

ls -lh "$DRIVE_ROOT/baseline/run0_week5_checkpoint"

echo

echo "=== Prediction line counts ==="
for f in \
  "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_val_200.jsonl" \
  "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl" \
  "$DRIVE_ROOT/baseline/run0_week5_checkpoint/predictions_test_200.jsonl"; do
  if [ -f "$f" ]; then
    wc -l "$f"
  else
    echo "[MISSING or intentionally skipped] $f"
  fi
done

echo

echo "=== Metrics preview ==="
for f in "$DRIVE_ROOT/baseline/run0_week5_checkpoint"/metrics_*.json; do
  [ -f "$f" ] || continue
  echo "--- $(basename "$f") ---"
  python - <<PY
import json
from pathlib import Path
p = Path("$f")
m = json.loads(p.read_text(encoding="utf-8"))
for k in ["n_prepared", "n_skipped", "strict_wer", "strict_cer", "normalized_wer", "normalized_cer", "runtime_seconds_per_sample"]:
    print(k, m.get(k))
PY
done

total 468K
-rw------- 1 root root 2.0K Jun 16 05:59 BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md
drwx------ 2 root root 4.0K Jun 16 05:47 mentor_asr_benchmark
-rw------- 1 root root  936 Jun 16 05:59 metrics_hard_200.json
-rw------- 1 root root  930 Jun 16 05:13 metrics_val_200.json
-rw------- 1 root root 229K Jun 16 05:59 predictions_hard_200.jsonl
-rw------- 1 root root 226K Jun 16 05:13 predictions_val_200.jsonl
-rw------- 1 root root  924 Jun 16 05:01 smoke_metrics_val_3.json
-rw------- 1 root root 4.0K Jun 16 05:01 smoke_predictions_val_3.jsonl

=== Prediction line counts ===
200 /content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/predictions_val_200.jsonl
200 /content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/predictions_hard_200.jsonl
[MISSING or intentionally skipped] /content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint/predictions_test_200.jsonl

=== Metrics preview ===
--- metrics_hard_200.json ---
n_prepared

### Phase 3N — Bundle artifact cần tải về local sau Phase 3

Chạy cell này nếu bạn muốn tải kết quả Phase 3 về máy local. Bundle này không chứa checkpoint và không chứa work dir tạm.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

OUT="/content/phase3_vimedcss_run0_baseline_artifacts"
rm -rf "$OUT"
mkdir -p "$OUT"

mkdir -p "$OUT/baseline"
if [ -d "$DRIVE_ROOT/baseline/run0_week5_checkpoint" ]; then
  cp -r "$DRIVE_ROOT/baseline/run0_week5_checkpoint" "$OUT/baseline/"
fi
if [ -d "$DRIVE_ROOT/baseline/run0_original_phowhisper" ]; then
  cp -r "$DRIVE_ROOT/baseline/run0_original_phowhisper" "$OUT/baseline/"
fi

mkdir -p "$OUT/scripts/asr_vimedcss"
cp "$PROJECT_ROOT/scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py" "$OUT/scripts/asr_vimedcss/"
cp "$PROJECT_ROOT/scripts/asr_vimedcss/03_make_run0_baseline_report.py" "$OUT/scripts/asr_vimedcss/"

mkdir -p "$OUT/manifests/run0"
cp "$PROJECT_ROOT/experiments/asr/vimedcss/manifests/run0/"*.jsonl "$OUT/manifests/run0/" 2>/dev/null || true

tar -czf /content/phase3_vimedcss_run0_baseline_artifacts.tar.gz -C /content phase3_vimedcss_run0_baseline_artifacts

ls -lh /content/phase3_vimedcss_run0_baseline_artifacts.tar.gz
find "$OUT" -maxdepth 4 -type f | sort

-rw-r--r-- 1 root root 125K Jun 16 05:59 /content/phase3_vimedcss_run0_baseline_artifacts.tar.gz
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/asr_benchmark_hard_200.txt
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/asr_benchmark_val_200.txt
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_prediction.tsv
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/hard_200_reference.tsv
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/val_200_prediction.tsv
/content/phase3_vimedcss_run0_baseline_artifacts/baseline/run0_week5_checkpoint/mentor_asr_benchmark/val_200_reference.tsv
/content/phase3_vimedcss_run0_b

### Phase 3O — Optional download Phase 3 bundle về máy local

In [ ]:
# # Chỉ chạy cell này nếu muốn tải bundle Phase 3 về máy local.
# from google.colab import files
# files.download('/content/phase3_vimedcss_run0_baseline_artifacts.tar.gz')

In [ ]:
from google.colab import files
import os

phase3_bundle = '/content/phase3_vimedcss_run0_baseline_artifacts.tar.gz'

if os.path.exists(phase3_bundle):
    print(f'Đang khởi tạo tải xuống: {phase3_bundle}')
    files.download(phase3_bundle)
else:
    print('Không tìm thấy tệp bundle Phase 3. Vui lòng chạy lại cell 35a14bdf để tạo lại.')

Đang khởi tạo tải xuống: /content/phase3_vimedcss_run0_baseline_artifacts.tar.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Phase 3P — Cleanup Colab work dir

Chỉ xóa work dir trong `/content`, không xóa Drive.

In [ ]:
%%bash
set -euo pipefail

source /content/phase3_env.sh

du -sh "$WORK_ROOT" || true

rm -rf "$WORK_ROOT/baseline_val"
rm -rf "$WORK_ROOT/baseline_hard"
rm -rf "$WORK_ROOT/baseline_test"
rm -rf "$WORK_ROOT/baseline_original_val"
rm -rf "$WORK_ROOT/smoke_baseline_val"

du -sh "$WORK_ROOT" || true

In [ ]:
import os
from pathlib import Path

drive_baseline_path = Path('/content/drive/MyDrive/clinical_asr_vimedcss/baseline')
week5_dir = drive_baseline_path / 'run0_week5_checkpoint'
original_dir = drive_baseline_path / 'run0_original_phowhisper'

def check_artifacts(directory, expected_files):
    print(f"\nChecking: {directory}")
    if not directory.exists():
        print(" [!] Directory does not exist.")
        return

    for f in expected_files:
        p = directory / f
        status = "[OK]" if p.exists() else "[MISSING]"
        print(f" {status} {f}")

# Check Week 5 Checkpoint Artifacts
week5_files = [
    "predictions_val_200.jsonl",
    "predictions_hard_200.jsonl",
    "predictions_test_200.jsonl",
    "metrics_val_200.json",
    "metrics_hard_200.json",
    "metrics_test_200.json",
    "BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md"
]
check_artifacts(week5_dir, week5_files)

# Check Optional Original PhoWhisper Artifacts
original_files = [
    "predictions_val_200.jsonl",
    "metrics_val_200.json",
    "BASELINE_RUN0_ORIGINAL_PHOWHISPER_REPORT.md"
]
check_artifacts(original_dir, original_files)


Checking: /content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint
 [OK] predictions_val_200.jsonl
 [OK] predictions_hard_200.jsonl
 [MISSING] predictions_test_200.jsonl
 [OK] metrics_val_200.json
 [OK] metrics_hard_200.json
 [MISSING] metrics_test_200.json
 [OK] BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md

Checking: /content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_original_phowhisper
 [MISSING] predictions_val_200.jsonl
 [MISSING] metrics_val_200.json
 [MISSING] BASELINE_RUN0_ORIGINAL_PHOWHISPER_REPORT.md


## Phase 3.II - Mentor-provided test benchmark

### Phase 3.II.0 — Mục tiêu và nguyên tắc xử lý mentor test set

---



Bộ `mentor_test_v1` là **external benchmark set do mentor cung cấp**, tách biệt với ViMedCSS subset tự materialize ở Phase 2.

Cấu trúc Drive đã chốt:

```text
/content/drive/MyDrive/clinical_asr_vimedcss/
└── external_benchmarks/
    └── mentor_test_v1/
        ├── raw/
        │   └── testset.zip
        ├── reference/
        ├── outputs/
        │   └── run0_week5_checkpoint/
        └── reports/
```

Nguyên tắc:

```text
1. Không trộn testset.zip vào subset_archives/.
2. Không unzip trực tiếp ra Drive nếu không cần.
3. Extract tạm vào /content/mentor_test_work/.
4. Chuẩn hóa thành:
   - mentor_test_manifest.jsonl
   - mentor_test_reference.tsv
5. Chạy ASR inference bằng Week 5 checkpoint.
6. Chạy script benchmark của sếp:
   scripts/asr_vimedcss/asr_benchmark.py
7. Lưu prediction, metrics, benchmark report về Drive.
```


In [7]:
%%bash
set -euo pipefail

# Reuse Phase 3 env if it exists.
if [ -f /content/phase3_env.sh ]; then
  source /content/phase3_env.sh
fi

PROJECT_ROOT="${PROJECT_ROOT:-/content/Clinical-Ambient-Documentation-Assistant}"
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/clinical_asr_vimedcss}"
WORK_ROOT="${WORK_ROOT:-/content/vimedcss_work}"
MODEL="${MODEL:-/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint}"

MENTOR_ROOT="$DRIVE_ROOT/external_benchmarks/mentor_test_v1"
MENTOR_ZIP="$MENTOR_ROOT/raw/testset.zip"
MENTOR_WORK="/content/mentor_test_work"
MENTOR_EXTRACT_DIR="$MENTOR_WORK/extracted"
MENTOR_REF_DIR="$MENTOR_ROOT/reference"
MENTOR_OUTPUT_ROOT="$MENTOR_ROOT/outputs"
MENTOR_REPORT_DIR="$MENTOR_ROOT/reports"

mkdir -p "$MENTOR_ROOT/raw"
mkdir -p "$MENTOR_REF_DIR"
mkdir -p "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint"
mkdir -p "$MENTOR_REPORT_DIR"
mkdir -p "$MENTOR_WORK"

cat > /content/mentor_test_env.sh <<EOF
export PROJECT_ROOT="$PROJECT_ROOT"
export DRIVE_ROOT="$DRIVE_ROOT"
export WORK_ROOT="$WORK_ROOT"
export MODEL="$MODEL"
export MENTOR_ROOT="$MENTOR_ROOT"
export MENTOR_ZIP="$MENTOR_ZIP"
export MENTOR_WORK="$MENTOR_WORK"
export MENTOR_EXTRACT_DIR="$MENTOR_EXTRACT_DIR"
export MENTOR_REF_DIR="$MENTOR_REF_DIR"
export MENTOR_OUTPUT_ROOT="$MENTOR_OUTPUT_ROOT"
export MENTOR_REPORT_DIR="$MENTOR_REPORT_DIR"
EOF

echo "[OK] mentor_test_env.sh"
cat /content/mentor_test_env.sh


[OK] mentor_test_env.sh
export PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"
export DRIVE_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss"
export WORK_ROOT="/content/vimedcss_work"
export MODEL="/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint"
export MENTOR_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1"
export MENTOR_ZIP="/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/raw/testset.zip"
export MENTOR_WORK="/content/mentor_test_work"
export MENTOR_EXTRACT_DIR="/content/mentor_test_work/extracted"
export MENTOR_REF_DIR="/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference"
export MENTOR_OUTPUT_ROOT="/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/outputs"
export MENTOR_REPORT_DIR="/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reports"


### Phase 3.II.1 — Kiểm tra input zip, checkpoint và benchmark script

Cell này chỉ kiểm tra điều kiện đầu vào. Nếu fail ở đây thì chưa chạy inference.


In [8]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

echo "=== Check mentor zip ==="
if [ ! -f "$MENTOR_ZIP" ]; then
  echo "[FAIL] Missing mentor zip: $MENTOR_ZIP"
  exit 1
fi
ls -lh "$MENTOR_ZIP"

echo
echo "=== Check Week 5 checkpoint ==="
if [ ! -d "$MODEL" ]; then
  echo "[FAIL] MODEL directory does not exist: $MODEL"
  echo "Fix MODEL in /content/mentor_test_env.sh or /content/phase3_env.sh before continuing."
  exit 1
fi
find "$MODEL" -maxdepth 1 -type f | sort | head -50

echo
echo "=== Check mentor benchmark script ==="
BENCH_SCRIPT="$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"
if [ ! -f "$BENCH_SCRIPT" ]; then
  echo "[FAIL] Missing benchmark script: $BENCH_SCRIPT"
  exit 1
fi
chmod +x "$BENCH_SCRIPT"
echo "[OK] $BENCH_SCRIPT"

echo
echo "=== Check GPU ==="
python - <<'PY'
import torch
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("[WARN] CUDA unavailable. Inference can run on CPU but will be slow.")
PY


=== Check mentor zip ===
-rw------- 1 root root 1010M Jun 16 09:48 /content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/raw/testset.zip

=== Check Week 5 checkpoint ===
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/generation_config.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/.gitattributes
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/medical_errors_dev.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/metrics_dev_project_wer.json
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/model.safetensors
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/predictions_dev.jsonl
/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint/preprocessor_config.json
/con

### Phase 3.II.2 — Preview cấu trúc zip trước khi extract

Mục tiêu là nhìn nhanh zip sếp đưa có những file gì: audio, transcript/reference, metadata.


In [10]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

PREVIEW_TXT="$MENTOR_REPORT_DIR/testset_zip_preview.txt"

echo "=== ZIP preview: first 120 lines ===" | tee "$PREVIEW_TXT"

# Temporarily disable pipefail for the following command to avoid SIGPIPE error
set +o pipefail
unzip -l "$MENTOR_ZIP" | head -120 | tee -a "$PREVIEW_TXT"
set -o pipefail # Re-enable pipefail

echo
echo "[DONE] zip preview saved to:"
echo "$PREVIEW_TXT"

=== ZIP preview: first 120 lines ===
Archive:  /content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/raw/testset.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-06-12 02:01   testset/
        0  2026-06-12 01:54   testset/audio/
   224078  2026-06-12 01:48   testset/audio/VietMed_004_1286.0000_1293.0000.wav
   288078  2026-06-12 01:48   testset/audio/test_Med_CS-3818-45.wav
   256078  2026-06-12 01:48   testset/audio/VietMed_017_52.0000_60.0000.wav
   103140  2026-06-12 01:48   testset/audio/bacsidatnhkhoavitadoc_2_20.wav
   192078  2026-06-12 01:48   testset/audio/test_Med_CS-3510-12.wav
   160078  2026-06-12 01:48   testset/audio/hard_Med_CS-95-4.wav
   224078  2026-06-12 01:48   testset/audio/VietMed_018_2275.0000_2282.0000.wav
   224078  2026-06-12 01:48   testset/audio/hard_Med_CS-2754-23.wav
   160078  2026-06-12 01:48   testset/audio/VietMed_024_3369.0000_3374.0000.wav
   256078  2026-06-12 01:48   testset/audio

### Phase 3.II.3 — Extract zip tạm vào `/content`

Không extract ra Drive để tránh chậm và tránh nhân đôi dung lượng trên Drive.


In [12]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

rm -rf "$MENTOR_EXTRACT_DIR"
mkdir -p "$MENTOR_EXTRACT_DIR"

echo "[INFO] Extracting:"
echo "$MENTOR_ZIP"
echo "[INFO] To:"
echo "$MENTOR_EXTRACT_DIR"

unzip -q "$MENTOR_ZIP" -d "$MENTOR_EXTRACT_DIR"

echo "[DONE] Extracted."

echo
echo "=== Extracted files preview ==="
# Disable pipefail temporarily to allow 'head' to close the pipe without triggering an error
set +o pipefail
find "$MENTOR_EXTRACT_DIR" -maxdepth 4 -type f | head -120
set -o pipefail

echo
echo "=== Extracted size ==="
du -sh "$MENTOR_EXTRACT_DIR"

[INFO] Extracting:
/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/raw/testset.zip
[INFO] To:
/content/mentor_test_work/extracted
[DONE] Extracted.

=== Extracted files preview ===
/content/mentor_test_work/extracted/testset/transcript.txt
/content/mentor_test_work/extracted/testset/audio/test_Med_CS-2324-7.wav
/content/mentor_test_work/extracted/testset/audio/test_Med_CS-2718-15.wav
/content/mentor_test_work/extracted/testset/audio/test_Med_CS-3743-8.wav
/content/mentor_test_work/extracted/testset/audio/VietMed_024_872.0000_880.0000.wav
/content/mentor_test_work/extracted/testset/audio/test_Med_CS-3525-16.wav
/content/mentor_test_work/extracted/testset/audio/VietMed_002_361.0000_367.0000.wav
/content/mentor_test_work/extracted/testset/audio/VietMed_019_1430.0000_1436.0000.wav
/content/mentor_test_work/extracted/testset/audio/VietMed_028_3615.0000_3621.0000.wav
/content/mentor_test_work/extracted/testset/audio/VietMed_023_3752.0000_3759.0000.wav
/content

### Phase 3.II.4 — Tạo script chuẩn hóa mentor test set

Script này tự động:

```text
1. Quét audio trong thư mục extracted.
2. Tìm file reference/transcription trong zip.
3. Thử đọc các format phổ biến: .tsv, .txt, .csv, .jsonl, .json.
4. Match reference với audio bằng basename/stem.
5. Xuất:
   - mentor_test_manifest.jsonl
   - mentor_test_reference.tsv
   - mentor_test_prepare_summary.json
   - MENTOR_TEST_PREPARE_REPORT.md
```

Nếu auto-detect sai, ta sẽ sửa bằng cách chỉ định `MENTOR_REFERENCE_FILE` thủ công ở cell sau.


In [13]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
mkdir -p "$PROJECT_ROOT/scripts/asr_vimedcss"

cat > "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_prepare_manifest.py" <<'PY'
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Prepare mentor-provided ASR test set for inference and benchmark.

Input:
- extracted_root: folder extracted from mentor testset.zip
- optional reference_file: known transcript/reference file

Output:
- mentor_test_manifest.jsonl with audio path + reference_text
- mentor_test_reference.tsv in filename<TAB>text format
- inventory/report/summary files
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import re
from pathlib import Path
from typing import Any

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".aac", ".wma"}
TEXT_EXTS = {".txt", ".tsv", ".csv", ".jsonl", ".json"}

KEY_FIELDS = [
    "filename", "file_name", "file", "audio", "audio_path", "path", "wav", "wav_path",
    "id", "sample_id", "segment_id", "utt_id", "utterance_id",
]
TEXT_FIELDS = [
    "text", "transcript", "transcription", "sentence", "reference", "reference_text",
    "segment_text", "label",
]


def norm_space(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s).strip())


def clean_text(s: Any) -> str:
    return norm_space(str(s).replace("\t", " ").replace("\n", " "))


def key_variants(key: str) -> list[str]:
    raw = str(key).strip().replace("\\", "/")
    p = Path(raw)
    name = p.name
    stem = p.stem
    variants = {
        raw,
        raw.lower(),
        name,
        name.lower(),
        stem,
        stem.lower(),
    }
    # Also remove common audio extensions from raw string.
    lowered = raw.lower()
    for ext in AUDIO_EXTS:
        if lowered.endswith(ext):
            variants.add(raw[: -len(ext)])
            variants.add(raw[: -len(ext)].lower())
    return [v for v in variants if v]


def add_index(index: dict[str, list[Path]], key: str, path: Path) -> None:
    for v in key_variants(key):
        index.setdefault(v.lower(), []).append(path)


def build_audio_index(audio_files: list[Path], root: Path) -> dict[str, list[Path]]:
    index: dict[str, list[Path]] = {}
    for p in audio_files:
        add_index(index, p.name, p)
        add_index(index, p.stem, p)
        try:
            add_index(index, str(p.relative_to(root)), p)
        except Exception:
            pass
    return index


def find_audio_for_key(key: str, index: dict[str, list[Path]]) -> Path | None:
    for v in key_variants(key):
        hits = index.get(v.lower())
        if hits:
            return sorted(hits)[0]
    return None


def read_delimited(path: Path, delimiter: str | None = None) -> dict[str, str]:
    out: dict[str, str] = {}

    with path.open("r", encoding="utf-8-sig", errors="replace", newline="") as f:
        sample = f.read(4096)
        f.seek(0)

        if delimiter is None:
            if path.suffix.lower() == ".tsv" or "\t" in sample:
                delimiter = "\t"
            else:
                delimiter = ","

        reader = csv.reader(f, delimiter=delimiter)
        rows = [row for row in reader if row and any(str(x).strip() for x in row)]

    if not rows:
        return out

    header = [x.strip().lower() for x in rows[0]]
    has_header = any(h in KEY_FIELDS + TEXT_FIELDS for h in header)

    if has_header:
        key_idx = next((i for i, h in enumerate(header) if h in KEY_FIELDS), 0)
        text_idx = next((i for i, h in enumerate(header) if h in TEXT_FIELDS), 1 if len(header) > 1 else 0)
        data_rows = rows[1:]
    else:
        key_idx = 0
        text_idx = 1 if len(rows[0]) > 1 else 0
        data_rows = rows

    for row in data_rows:
        if len(row) <= max(key_idx, text_idx):
            continue
        key = norm_space(row[key_idx])
        text = clean_text(row[text_idx])
        if key and text:
            out[key] = text

    return out


def read_plain_text(path: Path) -> dict[str, str]:
    # Preferred format: filename<TAB>text.
    out: dict[str, str] = {}
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]

    tab_lines = [line for line in lines if "\t" in line]
    if tab_lines:
        for line in tab_lines:
            key, text = line.split("\t", 1)
            key = norm_space(key)
            text = clean_text(text)
            if key and text:
                out[key] = text
        return out

    # Fallback: first whitespace token is filename/id, rest is transcript.
    for line in lines:
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            key, text = parts
            if key and text:
                out[norm_space(key)] = clean_text(text)

    return out


def read_jsonl(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if not isinstance(row, dict):
                continue
            key = next((row.get(k) for k in KEY_FIELDS if row.get(k)), None)
            text = next((row.get(k) for k in TEXT_FIELDS if row.get(k)), None)
            if key and text:
                out[norm_space(key)] = clean_text(text)
    return out


def read_json_file(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    try:
        data = json.loads(path.read_text(encoding="utf-8-sig", errors="replace"))
    except Exception:
        return out

    if isinstance(data, dict):
        # Case 1: {filename: text}
        if all(isinstance(v, str) for v in data.values()):
            return {norm_space(k): clean_text(v) for k, v in data.items() if str(k).strip() and str(v).strip()}

        # Case 2: {"data": [...]} or similar.
        for value in data.values():
            if isinstance(value, list):
                data = value
                break

    if isinstance(data, list):
        for row in data:
            if not isinstance(row, dict):
                continue
            key = next((row.get(k) for k in KEY_FIELDS if row.get(k)), None)
            text = next((row.get(k) for k in TEXT_FIELDS if row.get(k)), None)
            if key and text:
                out[norm_space(key)] = clean_text(text)

    return out


def load_reference(path: Path) -> dict[str, str]:
    suffix = path.suffix.lower()
    try:
        if suffix == ".jsonl":
            return read_jsonl(path)
        if suffix == ".json":
            return read_json_file(path)
        if suffix == ".csv":
            return read_delimited(path, delimiter=",")
        if suffix == ".tsv":
            return read_delimited(path, delimiter="\t")
        if suffix == ".txt":
            return read_plain_text(path)
    except Exception as exc:
        print(f"[WARN] failed to read {path}: {exc}")
    return {}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--extracted_root", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--reference_file", default=None)
    ap.add_argument("--min_matched", type=int, default=1)
    args = ap.parse_args()

    root = Path(args.extracted_root)
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not root.exists():
        raise FileNotFoundError(f"Extracted root does not exist: {root}")

    audio_files = sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS])
    text_files = sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in TEXT_EXTS])

    inventory_path = out_dir / "mentor_test_inventory.csv"
    with inventory_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["kind", "path", "size_bytes"])
        for p in audio_files:
            w.writerow(["audio", str(p), p.stat().st_size])
        for p in text_files:
            w.writerow(["text_candidate", str(p), p.stat().st_size])

    if not audio_files:
        raise RuntimeError("No audio files found in extracted mentor test set.")

    audio_index = build_audio_index(audio_files, root)

    candidate_files: list[Path]
    if args.reference_file:
        candidate_files = [Path(args.reference_file)]
    else:
        # Avoid huge accidental text files. 50 MB is enough for transcript metadata.
        candidate_files = [p for p in text_files if p.stat().st_size <= 50 * 1024 * 1024]

    candidates = []
    for p in candidate_files:
        refs = load_reference(p)
        matched = []
        unmatched = []
        for key, text in refs.items():
            audio = find_audio_for_key(key, audio_index)
            if audio is None:
                unmatched.append(key)
            else:
                matched.append((key, text, audio))

        candidates.append({
            "path": str(p),
            "n_refs": len(refs),
            "n_matched": len(matched),
            "n_unmatched": len(unmatched),
            "matched_preview": [(k, str(a)) for k, _, a in matched[:10]],
            "unmatched_preview": unmatched[:20],
            "_matched_rows": matched,
        })

    candidates_sorted = sorted(candidates, key=lambda x: (x["n_matched"], x["n_refs"]), reverse=True)

    if not candidates_sorted or candidates_sorted[0]["n_matched"] < args.min_matched:
        report = {
            "status": "FAIL_NO_REFERENCE_MATCH",
            "extracted_root": str(root),
            "n_audio_files": len(audio_files),
            "n_text_candidates": len(text_files),
            "candidate_summary": [
                {k: v for k, v in c.items() if not k.startswith("_")} for c in candidates_sorted[:20]
            ],
            "inventory_csv": str(inventory_path),
        }
        (out_dir / "mentor_test_prepare_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError("Could not find a reference/transcription file that matches audio filenames. Inspect mentor_test_inventory.csv and set --reference_file manually.")

    best = candidates_sorted[0]
    matched_rows = best["_matched_rows"]

    manifest_path = out_dir / "mentor_test_manifest.jsonl"
    reference_tsv = out_dir / "mentor_test_reference.tsv"

    with manifest_path.open("w", encoding="utf-8") as mf, reference_tsv.open("w", encoding="utf-8") as rf:
        for ref_key, text, audio_path in matched_rows:
            sample_id = audio_path.name
            row = {
                "sample_id": sample_id,
                "audio": str(audio_path),
                "reference_text": text,
                "original_ref_key": ref_key,
                "audio_basename": audio_path.name,
                "audio_stem": audio_path.stem,
                "source_reference_file": best["path"],
            }
            mf.write(json.dumps(row, ensure_ascii=False) + "\n")
            rf.write(f"{sample_id}\t{text}\n")

    summary = {
        "status": "PASS",
        "extracted_root": str(root),
        "selected_reference_file": best["path"],
        "n_audio_files": len(audio_files),
        "n_text_candidates": len(text_files),
        "n_refs_in_selected_file": best["n_refs"],
        "n_matched": best["n_matched"],
        "n_unmatched": best["n_unmatched"],
        "manifest_jsonl": str(manifest_path),
        "reference_tsv": str(reference_tsv),
        "inventory_csv": str(inventory_path),
        "candidate_summary": [
            {k: v for k, v in c.items() if not k.startswith("_")} for c in candidates_sorted[:20]
        ],
    }
    (out_dir / "mentor_test_prepare_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    md = []
    md.append("# Mentor Test Set Preparation Report")
    md.append("")
    md.append("## Status")
    md.append("")
    md.append("PASS")
    md.append("")
    md.append("## Selected reference file")
    md.append("")
    md.append(f"`{best['path']}`")
    md.append("")
    md.append("## Counts")
    md.append("")
    md.append(f"- Audio files found: {len(audio_files)}")
    md.append(f"- Text/reference candidates found: {len(text_files)}")
    md.append(f"- References in selected file: {best['n_refs']}")
    md.append(f"- Matched audio/reference rows: {best['n_matched']}")
    md.append(f"- Unmatched references: {best['n_unmatched']}")
    md.append("")
    md.append("## Outputs")
    md.append("")
    md.append(f"- Manifest JSONL: `{manifest_path}`")
    md.append(f"- Reference TSV: `{reference_tsv}`")
    md.append(f"- Inventory CSV: `{inventory_path}`")
    md.append("")
    md.append("## Notes")
    md.append("")
    md.append("The benchmark ID is the audio basename, matching the mentor benchmark format `filename<TAB>text`.")
    (out_dir / "MENTOR_TEST_PREPARE_REPORT.md").write_text("\n".join(md), encoding="utf-8")

    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()
PY

chmod +x "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_prepare_manifest.py"

echo "[OK] Created:"
echo "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_prepare_manifest.py"


[OK] Created:
/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/mentor_test_prepare_manifest.py


### Phase 3.II.5 — Chuẩn hóa mentor test set thành manifest/reference TSV

Mặc định script sẽ tự tìm file reference tốt nhất trong zip. Nếu auto-detect fail, xem `mentor_test_inventory.csv`, rồi chạy lại với biến `MENTOR_REFERENCE_FILE`.


In [14]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

# Optional manual override.
# Example:
# export MENTOR_REFERENCE_FILE="$MENTOR_EXTRACT_DIR/path/to/transcription.txt"
MENTOR_REFERENCE_FILE="${MENTOR_REFERENCE_FILE:-}"

CMD=(
  python "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_prepare_manifest.py"
  --extracted_root "$MENTOR_EXTRACT_DIR"
  --output_dir "$MENTOR_REF_DIR"
)

if [ -n "$MENTOR_REFERENCE_FILE" ]; then
  CMD+=(--reference_file "$MENTOR_REFERENCE_FILE")
fi

"${CMD[@]}"

echo
echo "=== Prepared files ==="
ls -lh "$MENTOR_REF_DIR"

echo
echo "=== Reference preview ==="
head -5 "$MENTOR_REF_DIR/mentor_test_reference.tsv" || true

echo
echo "=== Manifest preview ==="
head -1 "$MENTOR_REF_DIR/mentor_test_manifest.jsonl" | python -m json.tool


{
  "status": "PASS",
  "extracted_root": "/content/mentor_test_work/extracted",
  "selected_reference_file": "/content/mentor_test_work/extracted/testset/transcript.txt",
  "n_audio_files": 5980,
  "n_text_candidates": 1,
  "n_refs_in_selected_file": 5980,
  "n_matched": 5980,
  "n_unmatched": 0,
  "manifest_jsonl": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_manifest.jsonl",
  "reference_tsv": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_reference.tsv",
  "inventory_csv": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_inventory.csv",
  "candidate_summary": [
    {
      "path": "/content/mentor_test_work/extracted/testset/transcript.txt",
      "n_refs": 5980,
      "n_matched": 5980,
      "n_unmatched": 0,
      "matched_preview": [
        [
          "bacsidatnhkhoavitadoc_1_1.wav",
          "/content/mentor_

### Phase 3.II.6 — Xem báo cáo chuẩn hóa mentor test set

Nếu `n_matched` thấp bất thường, dừng lại và kiểm tra reference file trước khi chạy ASR inference.


In [15]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

echo "=== Prepare report ==="
cat "$MENTOR_REF_DIR/MENTOR_TEST_PREPARE_REPORT.md"

echo
echo "=== Prepare summary JSON ==="
python -m json.tool "$MENTOR_REF_DIR/mentor_test_prepare_summary.json" | head -200

echo
echo "=== Line counts ==="
wc -l "$MENTOR_REF_DIR/mentor_test_manifest.jsonl"
wc -l "$MENTOR_REF_DIR/mentor_test_reference.tsv"


=== Prepare report ===
# Mentor Test Set Preparation Report

## Status

PASS

## Selected reference file

`/content/mentor_test_work/extracted/testset/transcript.txt`

## Counts

- Audio files found: 5980
- Text/reference candidates found: 1
- References in selected file: 5980
- Matched audio/reference rows: 5980
- Unmatched references: 0

## Outputs

- Manifest JSONL: `/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_manifest.jsonl`
- Reference TSV: `/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_reference.tsv`
- Inventory CSV: `/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_inventory.csv`

## Notes

The benchmark ID is the audio basename, matching the mentor benchmark format `filename<TAB>text`.
=== Prepare summary JSON ===
{
    "status": "PASS",
    "extracted_root": "/content/mentor_test_work/extracted",
    "selected_re

### Phase 3.II.7 — Tạo script chạy ASR inference trên mentor manifest

Script này chạy model trên `mentor_test_manifest.jsonl`, xuất:

```text
predictions.jsonl
prediction.tsv
inference_metrics.json
```

Sau đó dùng `prediction.tsv` + `mentor_test_reference.tsv` để chạy benchmark của sếp.


In [16]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

cat > "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py" <<'PY'
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Run ASR inference on mentor test manifest and export benchmark-ready TSV."""

from __future__ import annotations

import argparse
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import torch
from transformers import pipeline


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["_line_no"] = line_no
            rows.append(row)
    return rows


def clean_tsv_cell(text: Any) -> str:
    return str(text if text is not None else "").replace("\t", " ").replace("\n", " ").strip()


def load_done_ids(pred_path: Path) -> set[str]:
    done = set()
    if not pred_path.exists():
        return done
    with pred_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            try:
                row = json.loads(line)
            except Exception:
                continue
            sid = row.get("sample_id")
            if sid:
                done.add(str(sid))
    return done


def export_prediction_tsv(pred_jsonl: Path, pred_tsv: Path) -> int:
    n = 0
    pred_tsv.parent.mkdir(parents=True, exist_ok=True)
    with pred_jsonl.open("r", encoding="utf-8", errors="replace") as f, pred_tsv.open("w", encoding="utf-8") as out:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            sid = clean_tsv_cell(row.get("sample_id") or row.get("audio_basename") or f"sample_{n:06d}")
            hyp = clean_tsv_cell(row.get("prediction_text", ""))
            out.write(f"{sid}\t{hyp}\n")
            n += 1
    return n


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--output_predictions", required=True)
    ap.add_argument("--output_prediction_tsv", required=True)
    ap.add_argument("--output_metrics", required=True)
    ap.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")
    ap.add_argument("--language", default="vi")
    ap.add_argument("--task", default="transcribe")
    ap.add_argument("--max_samples", type=int, default=None)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--model_label", default=None)
    args = ap.parse_args()

    manifest = Path(args.manifest)
    pred_jsonl = Path(args.output_predictions)
    pred_tsv = Path(args.output_prediction_tsv)
    metrics_path = Path(args.output_metrics)

    rows = read_jsonl(manifest)
    if args.max_samples is not None:
        rows = rows[: args.max_samples]

    if not rows:
        raise RuntimeError("No rows in mentor test manifest.")

    if args.device == "auto":
        device = 0 if torch.cuda.is_available() else -1
    elif args.device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable.")
        device = 0
    else:
        device = -1

    pred_jsonl.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.parent.mkdir(parents=True, exist_ok=True)

    done_ids = load_done_ids(pred_jsonl) if args.resume else set()
    mode = "a" if args.resume and pred_jsonl.exists() else "w"

    print(f"[INFO] rows={len(rows)} device={device} resume={args.resume} already_done={len(done_ids)}")
    print(f"[INFO] loading model={args.model}")

    asr = pipeline(
        "automatic-speech-recognition",
        model=args.model,
        tokenizer=args.model,
        feature_extractor=args.model,
        device=device,
    )

    started = time.time()
    n_run = 0
    n_error = 0

    with pred_jsonl.open(mode, encoding="utf-8") as out:
        for idx, row in enumerate(rows, start=1):
            sid = str(row.get("sample_id") or row.get("audio_basename") or f"sample_{idx:06d}")
            if sid in done_ids:
                continue

            audio = row.get("audio")
            ref = row.get("reference_text", "")
            t0 = time.time()
            err = None
            hyp = ""

            try:
                result = asr(str(audio), generate_kwargs={"language": args.language, "task": args.task})
                hyp = result.get("text", "") if isinstance(result, dict) else str(result)
            except Exception as exc:
                err = repr(exc)
                n_error += 1

            runtime = time.time() - t0

            pred_row = {
                "sample_id": sid,
                "audio": audio,
                "reference_text": ref,
                "prediction_text": hyp,
                "model": args.model,
                "model_label": args.model_label or args.model,
                "runtime_seconds": round(runtime, 4),
                "error": err,
                "original_ref_key": row.get("original_ref_key"),
                "source_reference_file": row.get("source_reference_file"),
            }
            out.write(json.dumps(pred_row, ensure_ascii=False) + "\n")
            out.flush()
            n_run += 1

            if n_run % 10 == 0:
                print(f"[INFO] processed_new={n_run} current_row={idx}/{len(rows)}")

    n_tsv = export_prediction_tsv(pred_jsonl, pred_tsv)

    metrics = {
        "created_at": now_iso(),
        "manifest": str(manifest),
        "model": args.model,
        "model_label": args.model_label or args.model,
        "n_manifest_rows": len(rows),
        "n_newly_processed": n_run,
        "n_prediction_tsv_rows": n_tsv,
        "n_errors_new": n_error,
        "resume": args.resume,
        "device": device,
        "total_runtime_seconds": round(time.time() - started, 4),
        "predictions_jsonl": str(pred_jsonl),
        "prediction_tsv": str(pred_tsv),
    }
    metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

    print(json.dumps(metrics, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()
PY

chmod +x "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"

echo "[OK] Created:"
echo "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"


[OK] Created:
/content/Clinical-Ambient-Documentation-Assistant/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py


### Phase 3.II.8 — Smoke test 3 samples bằng Week 5 checkpoint

Chạy 3 mẫu trước để kiểm tra model load được, audio đọc được, prediction tạo được.


In [17]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
cd "$PROJECT_ROOT"

OUT_DIR="$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/smoke_3"
mkdir -p "$OUT_DIR"

python "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"   --manifest "$MENTOR_REF_DIR/mentor_test_manifest.jsonl"   --model "$MODEL"   --output_predictions "$OUT_DIR/predictions_smoke_3.jsonl"   --output_prediction_tsv "$OUT_DIR/prediction_smoke_3.tsv"   --output_metrics "$OUT_DIR/inference_metrics_smoke_3.json"   --max_samples 3   --model_label "week5_vietmed_checkpoint"

echo
echo "=== Smoke predictions ==="
cat "$OUT_DIR/predictions_smoke_3.jsonl"


[INFO] rows=3 device=0 resume=False already_done=0
[INFO] loading model=/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint
{
  "created_at": "2026-06-16T12:22:36.602108+00:00",
  "manifest": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/reference/mentor_test_manifest.jsonl",
  "model": "/content/drive/MyDrive/clinical_asr_models/phowhisper_vietmed_week5_checkpoint",
  "model_label": "week5_vietmed_checkpoint",
  "n_manifest_rows": 3,
  "n_newly_processed": 3,
  "n_prediction_tsv_rows": 3,
  "n_errors_new": 0,
  "resume": false,
  "device": 0,
  "total_runtime_seconds": 8.0398,
  "predictions_jsonl": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/outputs/run0_week5_checkpoint/smoke_3/predictions_smoke_3.jsonl",
  "prediction_tsv": "/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1/outputs/run0_week5_checkpoint/smoke_3/prediction_smoke_3.tsv"
}

=== Smoke predict

Loading weights: 100%|██████████| 947/947 [00:41<00:00, 22.78it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressT

### Phase 3.II.9 — Chạy full inference trên mentor test set bằng Week 5 checkpoint

Kết quả của cell này là prediction chính để chạy benchmark của sếp.

Nếu Colab bị ngắt giữa chừng, chạy lại cell này với `--resume` để tiếp tục.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
cd "$PROJECT_ROOT"

OUT_DIR="$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint"
mkdir -p "$OUT_DIR"

python "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"   --manifest "$MENTOR_REF_DIR/mentor_test_manifest.jsonl"   --model "$MODEL"   --output_predictions "$OUT_DIR/predictions.jsonl"   --output_prediction_tsv "$OUT_DIR/prediction.tsv"   --output_metrics "$OUT_DIR/inference_metrics.json"   --resume   --model_label "week5_vietmed_checkpoint"

echo
echo "=== Output files ==="
ls -lh "$OUT_DIR"

echo
echo "=== Prediction TSV rows ==="
wc -l "$OUT_DIR/prediction.tsv"

echo
echo "=== Inference metrics ==="
python -m json.tool "$OUT_DIR/inference_metrics.json" | head -120


### Phase 3.II.10 — Chạy benchmark script của sếp trên mentor test set

Input benchmark:

```text
reference:  mentor_test_reference.tsv
prediction: prediction.tsv
format:     filename<TAB>text
```

Nếu có `scripts/asr_vimedcss/viet_syllables.txt`, benchmark sẽ tính thêm CS-WER/N-WER.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
cd "$PROJECT_ROOT"

OUT_DIR="$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint"
BENCH_TXT="$OUT_DIR/asr_benchmark.txt"
REF_TSV="$MENTOR_REF_DIR/mentor_test_reference.tsv"
HYP_TSV="$OUT_DIR/prediction.tsv"
SYLLABLES="$PROJECT_ROOT/scripts/asr_vimedcss/viet_syllables.txt"

if [ ! -f "$REF_TSV" ]; then
  echo "[FAIL] Missing reference TSV: $REF_TSV"
  exit 1
fi

if [ ! -f "$HYP_TSV" ]; then
  echo "[FAIL] Missing prediction TSV: $HYP_TSV"
  exit 1
fi

if [ -f "$SYLLABLES" ]; then
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV"     --viet-syllables "$SYLLABLES" | tee "$BENCH_TXT"
else
  echo "[WARN] viet_syllables.txt not found. Running without CS-WER/N-WER."
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV" | tee "$BENCH_TXT"
fi

echo
echo "[DONE] Benchmark saved to:"
echo "$BENCH_TXT"


### Phase 3.II.11 — Tạo report tổng hợp mentor test benchmark

Report này là artifact chính để báo cáo với sếp cho checkpoint Week 5 trên test set sếp đưa.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

OUT_DIR="$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint"
REPORT="$MENTOR_REPORT_DIR/MENTOR_TEST_RUN0_WEEK5_BENCHMARK_REPORT.md"

cat > "$REPORT" <<EOF
# Mentor Test Benchmark — Run 0 Week 5 Checkpoint

## Objective

Evaluate the Week 5 VietMed PhoWhisper checkpoint on the mentor-provided external test set.

This benchmark is separate from the internal ViMedCSS Run 0 subset baseline.

## Input

- Raw zip: \`$MENTOR_ZIP\`
- Extracted work dir: \`$MENTOR_EXTRACT_DIR\`
- Normalized manifest: \`$MENTOR_REF_DIR/mentor_test_manifest.jsonl\`
- Reference TSV: \`$MENTOR_REF_DIR/mentor_test_reference.tsv\`
- Model: \`$MODEL\`

## Outputs

- Predictions JSONL: \`$OUT_DIR/predictions.jsonl\`
- Prediction TSV: \`$OUT_DIR/prediction.tsv\`
- Inference metrics: \`$OUT_DIR/inference_metrics.json\`
- Mentor benchmark text: \`$OUT_DIR/asr_benchmark.txt\`

## Preparation summary

\`\`\`json
$(python -m json.tool "$MENTOR_REF_DIR/mentor_test_prepare_summary.json")
\`\`\`

## Inference summary

\`\`\`json
$(python -m json.tool "$OUT_DIR/inference_metrics.json")
\`\`\`

## ASR benchmark output

\`\`\`text
$(cat "$OUT_DIR/asr_benchmark.txt")
\`\`\`

## Notes

- Benchmark format follows the mentor script requirement: \`filename<TAB>text\`.
- This result should be treated as the primary external benchmark from the mentor-provided test set.
- ViMedCSS subset baseline remains useful as an internal pipeline/control baseline, but it does not replace this benchmark.
EOF

echo "[DONE] Report created:"
echo "$REPORT"

echo
cat "$REPORT" | head -200


### Phase 3.II.12 — Bundle artifact mentor benchmark để tải về local

Bundle này **không chứa audio** và **không chứa zip 1GB**. Nó chỉ chứa manifest/reference/prediction/report/scripts.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

BUNDLE_DIR="/content/mentor_test_v1_run0_week5_benchmark_artifacts"
rm -rf "$BUNDLE_DIR"
mkdir -p "$BUNDLE_DIR"

mkdir -p "$BUNDLE_DIR/reference"
cp "$MENTOR_REF_DIR/mentor_test_manifest.jsonl" "$BUNDLE_DIR/reference/" 2>/dev/null || true
cp "$MENTOR_REF_DIR/mentor_test_reference.tsv" "$BUNDLE_DIR/reference/" 2>/dev/null || true
cp "$MENTOR_REF_DIR/mentor_test_prepare_summary.json" "$BUNDLE_DIR/reference/" 2>/dev/null || true
cp "$MENTOR_REF_DIR/MENTOR_TEST_PREPARE_REPORT.md" "$BUNDLE_DIR/reference/" 2>/dev/null || true

mkdir -p "$BUNDLE_DIR/outputs/run0_week5_checkpoint"
cp "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/predictions.jsonl" "$BUNDLE_DIR/outputs/run0_week5_checkpoint/" 2>/dev/null || true
cp "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/prediction.tsv" "$BUNDLE_DIR/outputs/run0_week5_checkpoint/" 2>/dev/null || true
cp "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/inference_metrics.json" "$BUNDLE_DIR/outputs/run0_week5_checkpoint/" 2>/dev/null || true
cp "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/asr_benchmark.txt" "$BUNDLE_DIR/outputs/run0_week5_checkpoint/" 2>/dev/null || true

mkdir -p "$BUNDLE_DIR/reports"
cp "$MENTOR_REPORT_DIR/testset_zip_preview.txt" "$BUNDLE_DIR/reports/" 2>/dev/null || true
cp "$MENTOR_REPORT_DIR/MENTOR_TEST_RUN0_WEEK5_BENCHMARK_REPORT.md" "$BUNDLE_DIR/reports/" 2>/dev/null || true

mkdir -p "$BUNDLE_DIR/scripts"
cp "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py" "$BUNDLE_DIR/scripts/" 2>/dev/null || true
cp "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_prepare_manifest.py" "$BUNDLE_DIR/scripts/" 2>/dev/null || true
cp "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py" "$BUNDLE_DIR/scripts/" 2>/dev/null || true

tar -czf /content/mentor_test_v1_run0_week5_benchmark_artifacts.tar.gz -C /content mentor_test_v1_run0_week5_benchmark_artifacts

echo "[DONE] Bundle created:"
ls -lh /content/mentor_test_v1_run0_week5_benchmark_artifacts.tar.gz

echo
echo "=== Bundle contents ==="
find "$BUNDLE_DIR" -maxdepth 4 -type f | sort


### Phase 3.II.13 — Optional download bundle về máy local

Chỉ chạy nếu muốn tải artifact về máy.


In [ ]:
# Optional: download mentor benchmark bundle to local machine.
# from google.colab import files
# files.download("/content/mentor_test_v1_run0_week5_benchmark_artifacts.tar.gz")


## Phase 4 hook — Re-run mentor benchmark after Run A / Run B

Các cell dưới đây dùng lại cùng mentor test set để benchmark model sau fine-tune.

Chỉ chạy sau khi đã có:

```text
$DRIVE_ROOT/finetune/runA_smoke_200/best_model
$DRIVE_ROOT/finetune/runB_mini_1000/best_model
```

Mục tiêu: sau mỗi Run A/B, chạy lại test set sếp đưa để so với Week 5 checkpoint.


### Phase 4 hook A — Benchmark Run A best_model trên mentor test set


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
cd "$PROJECT_ROOT"

RUN_MODEL="$DRIVE_ROOT/finetune/runA_smoke_200/best_model"
OUT_DIR="$MENTOR_OUTPUT_ROOT/runA_smoke_200"

if [ ! -d "$RUN_MODEL" ]; then
  echo "[SKIP] Run A best_model not found:"
  echo "$RUN_MODEL"
  exit 0
fi

mkdir -p "$OUT_DIR"

python "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"   --manifest "$MENTOR_REF_DIR/mentor_test_manifest.jsonl"   --model "$RUN_MODEL"   --output_predictions "$OUT_DIR/predictions.jsonl"   --output_prediction_tsv "$OUT_DIR/prediction.tsv"   --output_metrics "$OUT_DIR/inference_metrics.json"   --resume   --model_label "runA_smoke_200"

REF_TSV="$MENTOR_REF_DIR/mentor_test_reference.tsv"
HYP_TSV="$OUT_DIR/prediction.tsv"
BENCH_TXT="$OUT_DIR/asr_benchmark.txt"
SYLLABLES="$PROJECT_ROOT/scripts/asr_vimedcss/viet_syllables.txt"

if [ -f "$SYLLABLES" ]; then
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV"     --viet-syllables "$SYLLABLES" | tee "$BENCH_TXT"
else
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV" | tee "$BENCH_TXT"
fi

echo "[DONE] Run A mentor benchmark:"
echo "$BENCH_TXT"


### Phase 4 hook B — Benchmark Run B best_model trên mentor test set


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh
cd "$PROJECT_ROOT"

RUN_MODEL="$DRIVE_ROOT/finetune/runB_mini_1000/best_model"
OUT_DIR="$MENTOR_OUTPUT_ROOT/runB_mini_1000"

if [ ! -d "$RUN_MODEL" ]; then
  echo "[SKIP] Run B best_model not found:"
  echo "$RUN_MODEL"
  exit 0
fi

mkdir -p "$OUT_DIR"

python "$PROJECT_ROOT/scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py"   --manifest "$MENTOR_REF_DIR/mentor_test_manifest.jsonl"   --model "$RUN_MODEL"   --output_predictions "$OUT_DIR/predictions.jsonl"   --output_prediction_tsv "$OUT_DIR/prediction.tsv"   --output_metrics "$OUT_DIR/inference_metrics.json"   --resume   --model_label "runB_mini_1000"

REF_TSV="$MENTOR_REF_DIR/mentor_test_reference.tsv"
HYP_TSV="$OUT_DIR/prediction.tsv"
BENCH_TXT="$OUT_DIR/asr_benchmark.txt"
SYLLABLES="$PROJECT_ROOT/scripts/asr_vimedcss/viet_syllables.txt"

if [ -f "$SYLLABLES" ]; then
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV"     --viet-syllables "$SYLLABLES" | tee "$BENCH_TXT"
else
  python "$PROJECT_ROOT/scripts/asr_vimedcss/asr_benchmark.py"     --prediction "$HYP_TSV"     --transcription "$REF_TSV" | tee "$BENCH_TXT"
fi

echo "[DONE] Run B mentor benchmark:"
echo "$BENCH_TXT"


### Phase 4 hook C — Tạo bảng so sánh benchmark Week5 / Run A / Run B

Cell này gom các file `asr_benchmark.txt` thành một report ngắn.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

REPORT="$MENTOR_REPORT_DIR/MENTOR_TEST_PHASE4_COMPARISON_REPORT.md"

cat > "$REPORT" <<EOF
# Mentor Test Benchmark Comparison

This report compares ASR benchmark outputs on the mentor-provided test set.

## Models

- Week 5 checkpoint: \`$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/asr_benchmark.txt\`
- Run A smoke: \`$MENTOR_OUTPUT_ROOT/runA_smoke_200/asr_benchmark.txt\`
- Run B mini: \`$MENTOR_OUTPUT_ROOT/runB_mini_1000/asr_benchmark.txt\`

## Week 5 checkpoint

\`\`\`text
$(cat "$MENTOR_OUTPUT_ROOT/run0_week5_checkpoint/asr_benchmark.txt" 2>/dev/null || echo "MISSING")
\`\`\`

## Run A smoke 200

\`\`\`text
$(cat "$MENTOR_OUTPUT_ROOT/runA_smoke_200/asr_benchmark.txt" 2>/dev/null || echo "MISSING")
\`\`\`

## Run B mini 1000

\`\`\`text
$(cat "$MENTOR_OUTPUT_ROOT/runB_mini_1000/asr_benchmark.txt" 2>/dev/null || echo "MISSING")
\`\`\`

## Rule

Use this external mentor benchmark as the main reporting benchmark. Internal ViMedCSS validation can still be used for pipeline control, but it must not replace the mentor-provided test set.
EOF

echo "[DONE] Comparison report:"
echo "$REPORT"

cat "$REPORT" | head -240


### Phase 3.II / Phase 4 hook — Cleanup

Chỉ xóa work dir tạm trong `/content`. Không xóa zip gốc và không xóa outputs trên Drive.


In [ ]:
%%bash
set -euo pipefail

source /content/mentor_test_env.sh

echo "Before cleanup:"
du -sh "$MENTOR_WORK" || true

# Uncomment when you no longer need extracted audio in the current runtime.
# rm -rf "$MENTOR_WORK"

echo "After cleanup check:"
du -sh "$MENTOR_WORK" || true

echo "[NOTE] Zip and outputs remain on Drive:"
echo "$MENTOR_ROOT"


### 📦 Đóng gói và Tải về Toàn bộ Artifacts Phase 3
Cell này gom tất cả báo cáo và kết quả từ Phase 3.I và 3.II vào một file duy nhất.

#### 🛠 Kiểm tra tính toàn vẹn của Artifacts trước khi tải về
Cell này liệt kê lại một lần cuối các file quan trọng sẽ có trong gói download.

In [ ]:
import os
from pathlib import Path

DRIVE_PHASE3_I = Path('/content/drive/MyDrive/clinical_asr_vimedcss/baseline/run0_week5_checkpoint')
DRIVE_PHASE3_II = Path('/content/drive/MyDrive/clinical_asr_vimedcss/external_benchmarks/mentor_test_v1')

print("=== KIỂM TRA PHASE 3.I (ViMedCSS) ===")
if DRIVE_PHASE3_I.exists():
    for f in ['metrics_val_200.json', 'metrics_hard_200.json', 'BASELINE_RUN0_WEEK5_CHECKPOINT_REPORT.md']:
        status = "[OK]" if (DRIVE_PHASE3_I / f).exists() else "[MISSING]"
        print(f"{status} {f}")

print("\n=== KIỂM TRA PHASE 3.II (Mentor Test) ===")
if DRIVE_PHASE3_II.exists():
    # Kiểm tra reference
    for f in ['reference/mentor_test_reference.tsv', 'reports/MENTOR_TEST_RUN0_WEEK5_BENCHMARK_REPORT.md']:
        status = "[OK]" if (DRIVE_PHASE3_II / f).exists() else "[MISSING]"
        print(f"{status} {f}")
    # Kiểm tra outputs
    out_dir = DRIVE_PHASE3_II / 'outputs/run0_week5_checkpoint'
    for f in ['asr_benchmark.txt', 'prediction.tsv']:
        status = "[OK]" if (out_dir / f).exists() else "[MISSING]"
        print(f"{status} {f}")

#### 📥 Tiến hành đóng gói và Tải về
Nếu các file trên đều báo `[OK]`, bạn hãy chạy cell này để nhận file nén.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

ZIP_NAME = '/content/Clinical_ASR_Phase3_All_Artifacts.zip'
targets = {
    DRIVE_ROOT / 'baseline/run0_week5_checkpoint': 'phase3_I_vimedcss_baseline',
    DRIVE_ROOT / 'external_benchmarks/mentor_test_v1/reference': 'phase3_II_mentor_test/reference',
    DRIVE_ROOT / 'external_benchmarks/mentor_test_v1/outputs/run0_week5_checkpoint': 'phase3_II_mentor_test/outputs',
    DRIVE_ROOT / 'external_benchmarks/mentor_test_v1/reports': 'phase3_II_mentor_test/reports',
    PROJECT_ROOT / 'scripts/asr_vimedcss/03_run_phowhisper_eval_on_archive.py': 'scripts/03_run_phowhisper_eval_on_archive.py',
    PROJECT_ROOT / 'scripts/asr_vimedcss/03_make_run0_baseline_report.py': 'scripts/03_make_run0_baseline_report.py',
    PROJECT_ROOT / 'scripts/asr_vimedcss/asr_benchmark.py': 'scripts/asr_benchmark.py',
    PROJECT_ROOT / 'scripts/asr_vimedcss/mentor_test_prepare_manifest.py': 'scripts/mentor_test_prepare_manifest.py',
    PROJECT_ROOT / 'scripts/asr_vimedcss/mentor_test_run_inference_from_manifest.py': 'scripts/mentor_test_run_inference_from_manifest.py',
}

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as z:
    for src, arc_root in targets.items():
        if not src.exists(): continue
        if src.is_dir():
            for root, _, filenames in os.walk(src):
                for filename in filenames:
                    if any(x in filename for x in ['.bin', '.safetensors', '.tar.gz', '.zip']): continue
                    file_path = Path(root) / filename
                    rel_path = file_path.relative_to(src)
                    z.write(file_path, arcname=Path(arc_root) / rel_path)
        else:
            z.write(src, arcname=arc_root)

print(f"Thành công! File đã sẵn sàng: {ZIP_NAME}")
files.download(ZIP_NAME)